In [1]:
# === BOOTSTRAP (place as FIRST cell) ===
import os, importlib.util, warnings, re

# detect env + streamlit
HAS_STREAMLIT = importlib.util.find_spec("streamlit") is not None
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.exists("/kaggle")

# shared config (create if missing)
CONFIG = globals().get("CONFIG", {})

# safe defaults for auto-run
if IN_KAGGLE and not HAS_STREAMLIT:
    # clean CI run in Kaggle: no Streamlit UI
    CONFIG.update({
        "RUN_SPC": True,
        "RUN_UI": False,     # <- crucial: skip UI when Streamlit absent
        "RUN_GATE": True,
        "RUN_MEDS": True,
        "PHASE2_BUNDLE": True,
    })
else:
    # dev/demo defaults: UI on if Streamlit present
    CONFIG.setdefault("RUN_SPC", True)
    CONFIG.setdefault("RUN_UI", HAS_STREAMLIT)
    CONFIG.setdefault("RUN_GATE", True)
    CONFIG.setdefault("RUN_MEDS", True)
    CONFIG.setdefault("PHASE2_BUNDLE", True)

# optional override for local demos: export FORCE_UI=1
if os.environ.get("FORCE_UI") == "1" and HAS_STREAMLIT:
    CONFIG["RUN_UI"] = True

globals()["CONFIG"] = CONFIG

# Quiet known warnings (use regex *strings*, not re.compile)
import warnings

# sklearn pickle compat noise
warnings.filterwarnings(
    "ignore",
    message=r"Trying to unpickle estimator .* when using version .*",
    category=UserWarning,
    module=r"sklearn\.base",
)

# traitlets deprecation noise during nbconvert
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module=r"traitlets\.traitlets",
)

print("Bootstrap →", {
    "IN_KAGGLE": IN_KAGGLE,
    "HAS_STREAMLIT": HAS_STREAMLIT,
    **{k: CONFIG[k] for k in ("RUN_SPC","RUN_UI","RUN_GATE","RUN_MEDS","PHASE2_BUNDLE")}
})


Bootstrap → {'IN_KAGGLE': True, 'HAS_STREAMLIT': False, 'RUN_SPC': True, 'RUN_UI': False, 'RUN_GATE': True, 'RUN_MEDS': True, 'PHASE2_BUNDLE': True}


In [2]:
# === Force-wire model bundle (fixed path) ===
from pathlib import Path
import shutil, os

SRC = Path("/kaggle/input/ed-pipeline-bundle-ui/ed_phase2_model_thr_patched(1).joblib")
DST = Path("/kaggle/working") / SRC.name

if SRC.exists():
    DST.parent.mkdir(parents=True, exist_ok=True)
    if not DST.exists():
        shutil.copy2(SRC, DST)
    CONFIG["MODEL_BUNDLE_PATH"] = str(DST)   # allowed even if CONFIG is frozen
    print("✅ Using model bundle →", DST)
else:
    CONFIG["MODEL_BUNDLE_PATH"] = None
    CONFIG["PHASE2_BUNDLE"] = False
    print("⚠️ Bundle not found at:", SRC)


✅ Using model bundle → /kaggle/working/ed_phase2_model_thr_patched(1).joblib


In [3]:
# --- TOP-OF-NOTEBOOK GUARDS / PATHS BOOTSTRAP ---
import os, json, datetime as _dt

# Environment root
if os.path.exists("/kaggle/working"):
    BASE = "/kaggle/working"
elif os.path.exists("/content"):
    BASE = "/content"
else:
    BASE = "/mnt/data"
print("BASE =", BASE)
_p = lambda *p: os.path.join(BASE, *p)

# Single source of truth
CONFIG = {
    "RUN_PIPELINE": True,
    "RUN_UI": True,
    "RUN_MEDS": True,
    "RUN_SYNTH": False,
    "DATA_ROOT": BASE, 
    "EQUIPMENT_STATUS_PATH": _p("equipment_status.csv"),
    "EQUIPMENT_MOVES_LOG_PATH": _p("moves_log.csv"),
    "SOP_REGISTRY_PATH": _p("sop_registry.csv"),
    "QR_OUTPUT_DIR": _p("qrs"),
    "EVENT_LOG_PATH": _p("event_log.jsonl"),
}

# FS prep
os.makedirs(CONFIG["QR_OUTPUT_DIR"], exist_ok=True)
os.makedirs(os.path.dirname(CONFIG["EVENT_LOG_PATH"]), exist_ok=True)

# Logger that always uses current CONFIG (no stale capture)
def _append_event(ev: dict):
    ev = {"ts": _dt.datetime.utcnow().isoformat()+"Z", **(ev or {})}
    with open(CONFIG["EVENT_LOG_PATH"], "a", encoding="utf-8") as f:
        f.write(json.dumps(ev, ensure_ascii=False) + "\n")

print("EVENT_LOG_PATH →", CONFIG["EVENT_LOG_PATH"])


BASE = /kaggle/working
EVENT_LOG_PATH → /kaggle/working/event_log.jsonl


In [4]:
# Freeze CONFIG so later cells can't silently flip flags/paths.
_ALLOWED_CONFIG_KEYS_TO_CHANGE = {"MODEL_BUNDLE_PATH"}  # bundle loader may set this

class _FrozenConfig(dict):
    def __setitem__(self, k, v):
        if k in self and k not in _ALLOWED_CONFIG_KEYS_TO_CHANGE:
            raise RuntimeError(f"CONFIG is frozen; attempted to modify {k}. Keep a single source of truth.")
        super().__setitem__(k, v)
    def update(self, *args, **kwargs):
        if args:
            for k in args[0].keys():
                if k in self and k not in _ALLOWED_CONFIG_KEYS_TO_CHANGE:
                    raise RuntimeError(f"CONFIG is frozen; attempted to modify {k}.")
        for k in list(kwargs.keys()):
            if k in self and k not in _ALLOWED_CONFIG_KEYS_TO_CHANGE:
                raise RuntimeError(f"CONFIG is frozen; attempted to modify {k}.")
        return super().update(*args, **kwargs)

CONFIG = _FrozenConfig(CONFIG)
print("CONFIG frozen. Only allowed later change:", _ALLOWED_CONFIG_KEYS_TO_CHANGE)


CONFIG frozen. Only allowed later change: {'MODEL_BUNDLE_PATH'}


In [5]:
# --- Load model bundle from attached dataset → set CONFIG path ---
import os, shutil
DATASET = "ed-pipeline-bundle-ui"  # ← replace with your actual dataset slug
SRC = f"/kaggle/input/{DATASET}/ed_phase2_model_thr_patched(1).joblib"
DST = "/kaggle/working/ed_phase2_model_thr_patched(1).joblib"

assert os.path.exists(SRC), f"Not found: {SRC} (check dataset slug/file name)"
if not os.path.exists(DST):
    os.makedirs(os.path.dirname(DST), exist_ok=True)
    shutil.copy2(SRC, DST)

CONFIG["MODEL_BUNDLE_PATH"] = DST
CONFIG["SKIP_MODEL_DISCOVERY"] = True  # prevents fallbacks from overriding this
print("Bundle ready →", CONFIG["MODEL_BUNDLE_PATH"])


Bundle ready → /kaggle/working/ed_phase2_model_thr_patched(1).joblib


In [6]:

# -- Demo meds/ICU/ML flags (non-destructive update of CONFIG) --
CONFIG.setdefault("RUN_MEDS", True)
CONFIG.setdefault("RUN_SYNTH", True)
CONFIG.setdefault("SAVE_SYNTH", True)
CONFIG.setdefault("ALLERGIES_PATH", "/mnt/data/patient_allergies.json")
CONFIG.setdefault("MED_RULES_PATH", "/mnt/data/interaction_rules.json")


'/mnt/data/interaction_rules.json'

In [7]:

# WorkflowState invariant: defined before use; exposes required methods; timers/backlogs/alerts intact
from dataclasses import dataclass, field
from typing import Any, Dict, List
import pandas as pd

@dataclass
class WorkflowState:
    role: str
    state: Dict[str, Any] = field(default_factory=dict)
    timers: Dict[str, float] = field(default_factory=dict)
    backlogs: Dict[str, List[Any]] = field(default_factory=dict)
    alerts: List[str] = field(default_factory=list)

    def touch_now(self, ts: pd.Timestamp):
        # update an example timer
        self.timers["last_touch_epoch"] = float(ts.value) / 1e9

    def feature_dict(self) -> Dict[str, Any]:
        # safe, extendable
        fd = {
            "role": self.role,
            "since_vitals_min": self.state.get("since_vitals_min", 0.0),
            "alerts_count": len(self.alerts),
        }
        # pass through any extra scalar features
        for k,v in self.state.items():
            if isinstance(v,(int,float,str)) and k not in fd:
                fd[k]=v
        return fd

    def update_state_from_event(self, event: Dict[str, Any]):
        # naive: merge event into state; track backlog
        self.state.update(event)
        self.backlogs.setdefault("events", []).append(event)

    def apply_event_log(self, events: List[Dict[str, Any]]):
        for ev in events:
            self.update_state_from_event(ev)


In [8]:

# TinyCritics uses WorkflowState.feature_dict(); cold-start safe (no transform before fit)
import numpy as np

class TinyCritics:
    def __init__(self):
        self._fitted = False

    def fit(self, states, actions, rewards):
        # No-op fit to keep cold-start safe
        self._fitted = True
        return self

    def score(self, state: WorkflowState, actions: List[Dict[str,str]]):
        fd = state.feature_dict()
        n = len(actions)
        # Deterministic, bounded scores in [0,1]
        base = 0.5
        p = np.full(n, base, dtype=float)
        bonuses = np.zeros(n, dtype=float)
        uncertainty = np.full(n, 0.1, dtype=float)
        return p, bonuses, uncertainty


In [9]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from pathlib import Path
import pandas as pd, numpy as np
def _cfg(CONFIG: Any, key: str, default: Any=None) -> Any:
    try: return CONFIG.get(key, default)
    except Exception: return getattr(CONFIG, key, default) if hasattr(CONFIG, key) else default
def _ensure_parent(p: Path): p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)
@dataclass
class EquipmentRecord:
    equip_id: str; name: str=""; location: str=""; status: str=""; last_seen: Optional[str]=None; battery: Optional[float]=None; confidence: Optional[float]=None
    def to_row(self)->Dict[str,Any]: return {"equip_id":self.equip_id,"name":self.name,"location":self.location,"status":self.status,"last_seen":self.last_seen,"battery":self.battery,"confidence":self.confidence}
class EquipmentRepository:
    def __init__(self, status_csv: Path):
        self.status_csv=Path(status_csv); _ensure_parent(self.status_csv)
        if not self.status_csv.exists(): pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"]).to_csv(self.status_csv, index=False)
    def read(self)->pd.DataFrame:
        try: df=pd.read_csv(self.status_csv); 
        except Exception: return pd.DataFrame(columns=["equip_id","name","location","status","last_seen","battery","confidence"])
        if "equip_id" in df.columns: df["equip_id"]=df["equip_id"].astype(str); return df
    def upsert(self, rec: EquipmentRecord)->None:
        df=self.read(); row=pd.DataFrame([rec.to_row()])
        if df.empty: df=row
        else:
            mask=(df["equip_id"].astype(str)==str(rec.equip_id))
            if mask.any(): df.loc[mask,:]=row.values
            else: df=pd.concat([df,row], ignore_index=True)
        df.to_csv(self.status_csv, index=False)
class MovesLogRepository:
    def __init__(self, moves_csv: Path):
        self.moves_csv=Path(moves_csv); _ensure_parent(self.moves_csv)
        if not self.moves_csv.exists(): pd.DataFrame(columns=["equip_id","from","to","ts"]).to_csv(self.moves_csv, index=False)
    def append(self, equip_id:str, loc_from:str, loc_to:str, ts_iso:str)->None:
        row=pd.DataFrame([{"equip_id":equip_id,"from":loc_from,"to":loc_to,"ts":ts_iso}])
        try: prev=pd.read_csv(self.moves_csv) if self.moves_csv.exists() else None; df=pd.concat([prev,row], ignore_index=True) if prev is not None else row
        except Exception: df=row
        df.to_csv(self.moves_csv, index=False)
    def read(self)->pd.DataFrame:
        try: return pd.read_csv(self.moves_csv)
        except Exception: return pd.DataFrame(columns=["equip_id","from","to","ts"])
class SOPRegistry:
    def __init__(self, sop_csv: Path): self.sop_csv=Path(sop_csv); _ensure_parent(self.sop_csv)
    def read(self)->pd.DataFrame:
        if self.sop_csv.exists():
            try:
                df=pd.read_csv(self.sop_csv)
                for col in ["sop_id","title","pdf_path"]:
                    if col not in df.columns: df[col]=""
                return df
            except Exception: pass
        return pd.DataFrame(columns=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"])
class QRService:
    def __init__(self,out_dir:Path): 
        self.out_dir=Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
    def make(self,payload:str)->str:
        try:
            import qrcode
            fp=self.out_dir/f"qr_{abs(hash(payload))}.png"
            img=qrcode.make(payload); img.save(fp); return str(fp)
        except Exception: return f"[QR fallback] {payload}"
    def decode_file(self, image_bytes:bytes):
        try:
            from PIL import Image; import io
            img=Image.open(io.BytesIO(image_bytes))
            try:
                from pyzbar.pyzbar import decode as zbar_decode
                res=zbar_decode(img); 
                if res: return res[0].data.decode("utf-8","ignore")
            except Exception: pass
        except Exception: pass
        return None
class TrackerService:
    def __init__(self, equipment_repo:EquipmentRepository, moves_repo:MovesLogRepository, sop_registry:SOPRegistry, qr:QRService, config:Any):
        self.equipment_repo=equipment_repo; self.moves_repo=moves_repo; self.sop_registry=sop_registry; self.qr=qr; self.CONFIG=config
    @classmethod
    def from_config(cls, CONFIG:Any)->"TrackerService":
        return cls(EquipmentRepository(Path(_cfg(CONFIG,"EQUIPMENT_STATUS_PATH"))),
                   MovesLogRepository(Path(_cfg(CONFIG,"EQUIPMENT_MOVES_LOG_PATH"))),
                   SOPRegistry(Path(_cfg(CONFIG,"SOP_REGISTRY_PATH"))),
                   QRService(Path(_cfg(CONFIG,"QR_OUTPUT_DIR"))), CONFIG)
    def equipment_status(self)->pd.DataFrame: return self.equipment_repo.read()
    def log_move(self, equip_id:str, loc_from:str, loc_to:str)->None:
        ts_iso=pd.Timestamp.utcnow().isoformat(); df=self.equipment_repo.read()
        row=df[df["equip_id"].astype(str)==str(equip_id)]; name=row["name"].iloc[0] if not row.empty and "name" in row.columns else ""
        rec=EquipmentRecord(equip_id=equip_id,name=name,location=loc_to,status="moved",last_seen=ts_iso)
        self.equipment_repo.upsert(rec); self.moves_repo.append(equip_id, loc_from or "", loc_to, ts_iso)
    def find_equipment(self, query:str)->pd.DataFrame:
        q=(query or "").strip().lower(); df=self.equipment_repo.read()
        if not q: return df
        def hit(r): return any(q in str(r.get(k,"")).lower() for k in ["equip_id","name","location","status"])
        return df[df.apply(hit, axis=1)]
    def overdue_equipment(self, threshold_minutes:int=120)->pd.DataFrame:
        df=self.equipment_repo.read().copy()
        if df.empty or "last_seen" not in df.columns: return df.iloc[0:0]
        ts=pd.to_datetime(df["last_seen"],errors="coerce",utc=True); age_min=(pd.Timestamp.utcnow().tz_localize("UTC")-ts).dt.total_seconds()/60.0
        df["age_min"]=age_min; return df[age_min>float(threshold_minutes)].sort_values("age_min", ascending=False)
    def movement_stats(self)->Dict[str,pd.DataFrame]:
        log=self.moves_repo.read()
        if log.empty: return {"moves_per_equipment":log,"routes":log}
        per_eq=log.groupby("equip_id").size().reset_index(name="moves").sort_values("moves", ascending=False)
        routes=log.groupby(["from","to"]).size().reset_index(name="count").sort_values("count", ascending=False)
        return {"moves_per_equipment":per_eq,"routes":routes}
    def sop_table(self)->pd.DataFrame: return self.sop_registry.read()
    def search_sop(self, query:str)->pd.DataFrame:
        df=self.sop_registry.read().copy(); q=(query or "").strip().lower()
        if df.empty or not q: return df
        cols=[c for c in ["sop_id","title","keywords","version","status"] if c in df.columns]
        mask=df[cols].astype(str).apply(lambda col: col.str.lower().str.contains(q, na=False)).any(axis=1)
        return df[mask]
    def make_qr(self,payload:str)->str: return self.qr.make(payload)
    def decode_qr_bytes(self, image_bytes:bytes): return self.qr.decode_file(image_bytes)
print("tracker core ready")

tracker core ready


In [10]:
from pathlib import Path
from typing import Any, Dict, List
def _slugify(text:str)->str:
    import re; s=re.sub(r"[^a-zA-Z0-9]+","-",text.strip().lower()).strip("-"); return s or "sop"
def refresh_sop_registry(CONFIG: Any, base_url: str="https://sop-notaufnahme.de/sop/")->Dict[str,Any]:
    out_csv=Path(CONFIG["SOP_REGISTRY_PATH"]); pdf_dir=Path(CONFIG["DATA_ROOT"])/"sop_pdfs"; pdf_dir.mkdir(parents=True, exist_ok=True)
    try:
        import requests; from bs4 import BeautifulSoup
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":f"missing libs: {e}"}
    found=saved=errors=0; items=[]
    try:
        r=requests.get(base_url, timeout=15); r.raise_for_status(); soup=BeautifulSoup(r.text,"html.parser")
        links=sorted({a["href"] for a in soup.find_all("a", href=True) if "/product/" in a["href"] and a["href"].startswith("http")})
        for url in links:
            try:
                pr=requests.get(url, timeout=15); pr.raise_for_status(); ps=BeautifulSoup(pr.text,"html.parser")
                ttag=ps.find(["h1","h2"]); title=ttag.get_text(strip=True) if ttag else (ps.find("title").get_text(strip=True) if ps.find("title") else url)
                pdfs=[a["href"] for a in ps.find_all("a", href=True) if a["href"].lower().endswith(".pdf")]
                pdf_url=pdfs[0] if pdfs else None; sop_id=_slugify(title or url.split("/")[-2]); pdf_path=""
                if pdf_url:
                    try:
                        fn=sop_id+".pdf"; outp=pdf_dir/fn
                        with requests.get(pdf_url, stream=True, timeout=30) as dr:
                            dr.raise_for_status()
                            with open(outp,"wb") as f:
                                for chunk in dr.iter_content(8192):
                                    if chunk: f.write(chunk)
                        pdf_path=str(outp); saved+=1
                    except Exception:
                        errors+=1; pdf_path=pdf_url
                items.append({"sop_id":sop_id,"title":title or sop_id,"pdf_path":pdf_path,"version":"","status":"fetched" if pdf_path else "linked","keywords":"","checklist":"","source_url":url})
                found+=1
            except Exception: errors+=1; continue
    except Exception as e:
        return {"found":0,"saved":0,"errors":1,"error":str(e)}
    import pandas as pd
    try:
        if out_csv.exists(): df=pd.read_csv(out_csv)
        else: df=pd.DataFrame(columns=["sop_id","title","pdf_path","version","status","keywords","checklist","source_url"])
        df=df.copy()
        if df.empty: new_df=pd.DataFrame(items)
        else:
            df["sop_id"]=df["sop_id"].astype(str)
            for i in items:
                mask=(df["sop_id"]==str(i["sop_id"]))
                if mask.any():
                    for k,v in i.items():
                        if k in df.columns and (pd.isna(df.loc[mask,k]).all() or str(df.loc[mask,k].iloc[0]).strip()=="" or k in ["pdf_path","status","source_url"]):
                            df.loc[mask,k]=v
                else:
                    df=pd.concat([df, pd.DataFrame([i])], ignore_index=True)
            new_df=df
        new_df.to_csv(out_csv, index=False)
    except Exception as e:
        errors+=1
    return {"found":found,"saved":saved,"errors":errors,"csv":str(out_csv),"dir":str(pdf_dir)}
def load_priority_flows(json_path:str)->Dict[str,Any]:
    import json
    try:
        with open(json_path,"r",encoding="utf-8") as f: return json.load(f)
    except Exception: return {}
print("sop auto ready")

sop auto ready


In [11]:
def rule_hs_tnt(value):
    try: v = float(value)
    except Exception: return 0.50
    if v < 14: return 0.50
    if 14 <= v <= 51: return 0.20
    return 0.50
assert rule_hs_tnt(13.9)==0.50 and rule_hs_tnt(14.0)==0.20 and rule_hs_tnt(51.0)==0.20 and rule_hs_tnt(51.1)==0.50
print("troponin_rules ok")

troponin_rules ok


In [12]:
import importlib
def _try_import(name:str):
    try: return importlib.import_module(name)
    except Exception: return None
def run_icu_constraints(state):
    if not RUN_PIPELINE: return {}
    mod = _try_import("icu_constraints") or _try_import("modeling_icu_constraints")
    if mod and hasattr(mod,"compute_icu_flags"):
        try: return dict(mod.compute_icu_flags(state))
        except Exception: return {}
    return {}
def mesh_route_actions(state, actions):
    if not RUN_PIPELINE: return actions
    mod = _try_import("agent_mesh") or _try_import("ed_agent_mesh")
    if mod and hasattr(mod,"route"):
        try: return list(mod.route(state, actions))
        except Exception: return actions
    return actions
def trainer_fit_critic(critic, samples, y):
    if not RUN_PIPELINE: return critic
    mod = _try_import("trainer") or _try_import("ed_trainer")
    if mod and hasattr(mod,"fit_critic"):
        try: return mod.fit_critic(critic, samples, y)
        except Exception: return critic
    return critic
print("phase2 bridge ready")

phase2 bridge ready


In [13]:
def run_ui(tracker, get_state, get_actions, critic):
    import streamlit as st, pandas as pd, numpy as np
    st.set_page_config(page_title="ED Tracker — Full", layout="wide")
    st.title("ED Tracker — Core Ops (Full)")
    c0, c1, c2, c3 = st.columns([2,2,2,2])
    with c0:
        thresh = st.number_input("Overdue threshold (min)", min_value=5, max_value=720, value=120, step=5)
    with c1:
        if st.button("Refresh"): st.experimental_rerun()
    st.header("Equipment")
    eq_df = tracker.equipment_status()
    s1, s2 = st.columns([2,1])
    with s1:
        q = st.text_input("Find equipment (ID / name / location / status)", "")
        filt = tracker.find_equipment(q) if q else eq_df
        st.dataframe(filt, use_container_width=True, height=260)
    with s2:
        overdue = tracker.overdue_equipment(int(thresh))
        st.subheader("Overdue")
        if overdue.empty: st.write("None")
        else: st.dataframe(overdue[["equip_id","name","location","last_seen","age_min"]], use_container_width=True, height=200)
    st.markdown("**Update location / log move**")
    mc1, mc2, mc3, mc4 = st.columns([2,2,2,1])
    with mc1: sel_id = st.selectbox("Equipment ID", [""] + sorted(list(eq_df.get("equip_id", []))))
    with mc2: loc_from = st.text_input("From", "")
    with mc3: loc_to = st.text_input("To", "")
    with mc4:
        if st.button("Log move") and sel_id and loc_to:
            tracker.log_move(sel_id, loc_from, loc_to); st.success(f"Move logged: {sel_id} → {loc_to}")
    st.header("QR")
    qr_col1, qr_col2 = st.columns([2,2])
    with qr_col1:
        qr_txt = st.text_input("QR payload to generate", "")
        if st.button("Generate QR") and qr_txt:
            path = tracker.make_qr(qr_txt); st.write("QR saved to:", path)
    with qr_col2:
        st.write("Scan and update location")
        f = st.file_uploader("Upload QR image", type=["png","jpg","jpeg","webp"])
        manual_payload = st.text_input("Manual payload (fallback if decoding fails)", "")
        new_loc = st.text_input("New location (after scan)", "")
        if st.button("Scan & Update"):
            equip_payload = None
            if f is not None: equip_payload = tracker.decode_qr_bytes(f.read())
            if not equip_payload and manual_payload: equip_payload = manual_payload
            if equip_payload and new_loc:
                equip_id = equip_payload
                if "id=" in equip_payload:
                    try: equip_id = equip_payload.split("id=",1)[1].split("&",1)[0]
                    except Exception: equip_id = equip_payload
                tracker.log_move(str(equip_id), "", new_loc); st.success(f"Updated via payload. {equip_id} → {new_loc}")
            elif not new_loc: st.error("Provide a new location.")
            else: st.error("No QR payload detected (image or manual).")
    with st.expander("SOP auto-pull and flows", expanded=False):
        if st.button("Refresh SOPs from sop-notaufnahme.de"):
            res = refresh_sop_registry(CONFIG, base_url="https://sop-notaufnahme.de/sop/"); st.write(res)
        flows = load_priority_flows("/mnt/data/priority_flows.json")
        if flows:
            keys = sorted(list(flows.keys())); pickf = st.selectbox("Show flow", [""] + keys)
            if pickf:
                flow = flows[pickf]; st.subheader(flow.get("title", pickf))
                nodes = flow.get("nodes", []); edges = flow.get("edges", [])
                st.write("Nodes:", ", ".join([n.get("label", n.get("id","")) for n in nodes]))
                try:
                    import matplotlib.pyplot as plt
                    fig = plt.figure()
                    pos = {n["id"]:(i, 0) for i,n in enumerate(nodes)}
                    for n in nodes:
                        x,y = pos[n["id"]]; plt.scatter([x],[y]); plt.text(x,y+0.05,n.get("label", n["id"]), ha="center", rotation=45)
                    for a,b in edges:
                        xa,ya = pos.get(a,(0,0)); xb,yb = pos.get(b,(0,0)); plt.plot([xa,xb],[ya,yb])
                    plt.axis("off"); plt.title(flow.get("title", pickf)); st.pyplot(fig)
                except Exception: st.info("Graph display unavailable; showing list instead."); st.write(edges)
    st.header("SOPs")
    sop_q = st.text_input("Search SOPs (id/title/keywords)", "")
    sop_hits = tracker.search_sop(sop_q)
    if sop_hits.empty: st.info("No SOPs found.")
    else:
        st.dataframe(sop_hits[["sop_id","title","version","status"]], use_container_width=True, height=220)
        pick = st.selectbox("Open SOP", [""] + sop_hits["sop_id"].astype(str).tolist())
        if pick:
            row = sop_hits[sop_hits["sop_id"].astype(str)==pick].iloc[0]
            pdf = row.get("pdf_path","")
            if pdf: st.write("PDF path:", pdf)
            if "checklist" in sop_hits.columns and isinstance(row.get("checklist", None), str) and row["checklist"].strip():
                st.subheader("Checklist")
                steps = [s.strip() for s in row["checklist"].split("|") if s.strip()]
                completed = []
                for i, step in enumerate(steps, 1):
                    if st.checkbox(f"{i}. {step}", key=f"sop_{pick}_{i}"):
                        completed.append(i)
                st.caption(f"Completed {len(completed)}/{len(steps)} steps")
    st.header("Actions & Critic")
    state = get_state()
    if hasattr(state,"feature_dict"):
        feats = state.feature_dict(); since_v = feats.get("since_vitals_min", None)
        if since_v is not None:
            if since_v > 120: st.error(f"Lingering patient: since_vitals_min={since_v:.0f} > 120")
            else: st.success(f"Vitals recently checked: {since_v:.0f} min")
    if st.button("Mark vitals now") and hasattr(state,"touch_now"):
        state.touch_now(pd.Timestamp.utcnow()); st.success("Vitals timestamp updated.")
    actions = get_actions(state)
    if not actions: st.info("No actions available."); return
    p, benefit, burden = critic.score(state, actions)
    import pandas as pd, numpy as np
    view = pd.DataFrame({"id":[a.get("id") for a in actions],"label":[a.get("label") for a in actions],"p_accept":np.round(p,3),"benefit":np.round(benefit,3),"burden":np.round(burden,3)}).sort_values(["p_accept","benefit"], ascending=[False, False])
    st.dataframe(view, use_container_width=True, height=240)
    st.header("Equipment Movement Analytics")
    stats = tracker.movement_stats(); per_eq = stats["moves_per_equipment"]; routes = stats["routes"]
    if per_eq.empty: st.info("No movement data yet.")
    else:
        st.subheader("Moves per equipment"); st.dataframe(per_eq, use_container_width=True, height=240)
        try:
            import matplotlib.pyplot as plt
            fig = plt.figure(); x=per_eq["equip_id"].astype(str).tolist(); y=per_eq["moves"].tolist()
            plt.bar(x,y); plt.xticks(rotation=45, ha="right"); plt.title("Moves per Equipment"); st.pyplot(fig)
        except Exception: pass
        st.subheader("Top routes"); st.dataframe(routes, use_container_width=True, height=200)
print("ui ready")


ui ready


In [14]:

# Inline SOP surface and optional UI without external modules
from typing import Optional, Dict, Any
import csv, os
from pathlib import Path

def refresh_sop_registry(CONFIG: dict, base_url: Optional[str]) -> Dict[str, Any]:
    """
    Offline-safe: if base_url is provided and fetch works + CSV looks valid, overwrite the file.
    Otherwise, return a summary without raising. No sidecars.
    """
    path = Path(CONFIG["SOP_REGISTRY_PATH"])
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("w", newline="") as fp:
            csv.writer(fp).writerow(["id","title","url"])
    if base_url:
        try:
            import requests
            resp = requests.get(base_url, timeout=5)
            resp.raise_for_status()
            text = resp.text.strip()
            rows = [r.split(",") for r in text.splitlines()]
            if rows and len(rows[0])>=3:
                with path.open("w", newline="") as fp:
                    csv.writer(fp).writerows(rows)
                return {"ok": True, "rows": len(rows)-1}
        except Exception as e:
            return {"ok": False, "error": str(e)}
    return {"ok": False, "error": "No base_url or unexpected format"}

# Optional, guarded UI demo (equipment status preview uses the CSV directly)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd
        eq_path = Path(CONFIG["EQUIPMENT_STATUS_PATH"])
        if not eq_path.exists():
            eq_path.write_text("equipment_id,location,last_seen\n")
        btn = W.Button(description="Show equipment status")
        out = W.Output()
        def _on_click(_):
            with out:
                out.clear_output()
                try:
                    df = pd.read_csv(eq_path)
                except Exception:
                    df = pd.DataFrame(columns=["equipment_id","location","last_seen"])
                display(df)
        btn.on_click(_on_click)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable (optional):", e)


In [15]:
def run_ui(tracker, get_state, get_actions, critic):
    import streamlit as st, pandas as pd, numpy as np
    st.set_page_config(page_title="ED Tracker — Full", layout="wide")
    st.title("ED Tracker — Core Ops (Full)")
    c0, c1, c2, c3 = st.columns([2,2,2,2])
    with c0:
        thresh = st.number_input("Overdue threshold (min)", min_value=5, max_value=720, value=120, step=5)
    with c1:
        if st.button("Refresh"): st.experimental_rerun()
    st.header("Equipment")
    eq_df = tracker.equipment_status()
    s1, s2 = st.columns([2,1])
    with s1:
        q = st.text_input("Find equipment (ID / name / location / status)", "")
        filt = tracker.find_equipment(q) if q else eq_df
        st.dataframe(filt, use_container_width=True, height=260)
    with s2:
        overdue = tracker.overdue_equipment(int(thresh))
        st.subheader("Overdue")
        if overdue.empty: st.write("None")
        else: st.dataframe(overdue[["equip_id","name","location","last_seen","age_min"]], use_container_width=True, height=200)
    st.markdown("**Update location / log move**")
    mc1, mc2, mc3, mc4 = st.columns([2,2,2,1])
    with mc1: sel_id = st.selectbox("Equipment ID", [""] + sorted(list(eq_df.get("equip_id", []))))
    with mc2: loc_from = st.text_input("From", "")
    with mc3: loc_to = st.text_input("To", "")
    with mc4:
        if st.button("Log move") and sel_id and loc_to:
            tracker.log_move(sel_id, loc_from, loc_to); st.success(f"Move logged: {sel_id} → {loc_to}")
    st.header("QR")
    qr_col1, qr_col2 = st.columns([2,2])
    with qr_col1:
        qr_txt = st.text_input("QR payload to generate", "")
        if st.button("Generate QR") and qr_txt:
            path = tracker.make_qr(qr_txt); st.write("QR saved to:", path)
    with qr_col2:
        st.write("Scan and update location")
        f = st.file_uploader("Upload QR image", type=["png","jpg","jpeg","webp"])
        manual_payload = st.text_input("Manual payload (fallback if decoding fails)", "")
        new_loc = st.text_input("New location (after scan)", "")
        if st.button("Scan & Update"):
            equip_payload = None
            if f is not None: equip_payload = tracker.decode_qr_bytes(f.read())
            if not equip_payload and manual_payload: equip_payload = manual_payload
            if equip_payload and new_loc:
                equip_id = equip_payload
                if "id=" in equip_payload:
                    try: equip_id = equip_payload.split("id=",1)[1].split("&",1)[0]
                    except Exception: equip_id = equip_payload
                tracker.log_move(str(equip_id), "", new_loc); st.success(f"Updated via payload. {equip_id} → {new_loc}")
            elif not new_loc: st.error("Provide a new location.")
            else: st.error("No QR payload detected (image or manual).")
    with st.expander("SOP auto-pull and flows", expanded=False):
        if st.button("Refresh SOPs from sop-notaufnahme.de"):
            res = refresh_sop_registry(CONFIG, base_url="https://sop-notaufnahme.de/sop/"); st.write(res)
        flows = load_priority_flows("/mnt/data/priority_flows.json")
        if flows:
            keys = sorted(list(flows.keys())); pickf = st.selectbox("Show flow", [""] + keys)
            if pickf:
                flow = flows[pickf]; st.subheader(flow.get("title", pickf))
                nodes = flow.get("nodes", []); edges = flow.get("edges", [])
                st.write("Nodes:", ", ".join([n.get("label", n.get("id","")) for n in nodes]))
                try:
                    import matplotlib.pyplot as plt
                    fig = plt.figure()
                    pos = {n["id"]:(i, 0) for i,n in enumerate(nodes)}
                    for n in nodes:
                        x,y = pos[n["id"]]; plt.scatter([x],[y]); plt.text(x,y+0.05,n.get("label", n["id"]), ha="center", rotation=45)
                    for a,b in edges:
                        xa,ya = pos.get(a,(0,0)); xb,yb = pos.get(b,(0,0)); plt.plot([xa,xb],[ya,yb])
                    plt.axis("off"); plt.title(flow.get("title", pickf)); st.pyplot(fig)
                except Exception: st.info("Graph display unavailable; showing list instead."); st.write(edges)
    st.header("SOPs")
    sop_q = st.text_input("Search SOPs (id/title/keywords)", "")
    sop_hits = tracker.search_sop(sop_q)
    if sop_hits.empty: st.info("No SOPs found.")
    else:
        st.dataframe(sop_hits[["sop_id","title","version","status"]], use_container_width=True, height=220)
        pick = st.selectbox("Open SOP", [""] + sop_hits["sop_id"].astype(str).tolist())
        if pick:
            row = sop_hits[sop_hits["sop_id"].astype(str)==pick].iloc[0]
            pdf = row.get("pdf_path","")
            if pdf: st.write("PDF path:", pdf)
            if "checklist" in sop_hits.columns and isinstance(row.get("checklist", None), str) and row["checklist"].strip():
                st.subheader("Checklist")
                steps = [s.strip() for s in row["checklist"].split("|") if s.strip()]
                completed = []
                for i, step in enumerate(steps, 1):
                    if st.checkbox(f"{i}. {step}", key=f"sop_{pick}_{i}"):
                        completed.append(i)
                st.caption(f"Completed {len(completed)}/{len(steps)} steps")
    st.header("Actions & Critic")
    state = get_state()
    if hasattr(state,"feature_dict"):
        feats = state.feature_dict(); since_v = feats.get("since_vitals_min", None)
        if since_v is not None:
            if since_v > 120: st.error(f"Lingering patient: since_vitals_min={since_v:.0f} > 120")
            else: st.success(f"Vitals recently checked: {since_v:.0f} min")
    if st.button("Mark vitals now") and hasattr(state,"touch_now"):
        state.touch_now(pd.Timestamp.utcnow()); st.success("Vitals timestamp updated.")
    actions = get_actions(state)
    if not actions: st.info("No actions available."); return
    p, benefit, burden = critic.score(state, actions)
    import pandas as pd, numpy as np
    view = pd.DataFrame({"id":[a.get("id") for a in actions],"label":[a.get("label") for a in actions],"p_accept":np.round(p,3),"benefit":np.round(benefit,3),"burden":np.round(burden,3)}).sort_values(["p_accept","benefit"], ascending=[False, False])
    st.dataframe(view, use_container_width=True, height=240)
    st.header("Equipment Movement Analytics")
    stats = tracker.movement_stats(); per_eq = stats["moves_per_equipment"]; routes = stats["routes"]
    if per_eq.empty: st.info("No movement data yet.")
    else:
        st.subheader("Moves per equipment"); st.dataframe(per_eq, use_container_width=True, height=240)
        try:
            import matplotlib.pyplot as plt
            fig = plt.figure(); x=per_eq["equip_id"].astype(str).tolist(); y=per_eq["moves"].tolist()
            plt.bar(x,y); plt.xticks(rotation=45, ha="right"); plt.title("Moves per Equipment"); st.pyplot(fig)
        except Exception: pass
        st.subheader("Top routes"); st.dataframe(routes, use_container_width=True, height=200)
print("ui ready")


ui ready


In [16]:
# === Self-contained ML→OPS bridge (no external file needed) ===
import os, glob, json
import numpy as np, pandas as pd
from joblib import load

# 1) Locate bundle
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else "/mnt/data"
BUNDLE = CONFIG.get("MODEL_BUNDLE_PATH")

if not (BUNDLE and os.path.exists(BUNDLE)):
    candidates = []
    for root in [BASE, "/mnt/data", "/content"]:
        candidates += glob.glob(os.path.join(root, "**", "ed_phase2_model*_patched(1).joblib"), recursive=True)
        candidates += glob.glob(os.path.join(root, "**", "*.joblib"), recursive=True)
        candidates += glob.glob(os.path.join(root, "**", "*.pkl"), recursive=True)
    BUNDLE = next((p for p in candidates if os.path.exists(p)), None)
    if not BUNDLE:
        raise FileNotFoundError("Model bundle not found. Set CONFIG['MODEL_BUNDLE_PATH'].")

# 2) Load and expose scorer
b = load(BUNDLE)
pipe = b.get("pipeline") or b.get("model")
cal  = b.get("calibrator")  # may be None (CalibratedClassifierCV wraps calibration)
THR  = float(b.get("threshold", 0.5))
FEAT = b.get("features")    # list or None

def score_proba(df: pd.DataFrame) -> np.ndarray:
    X = df[FEAT] if FEAT else df
    p = pipe.predict_proba(X)[:, 1]
    return cal.transform(np.asarray(p)) if cal is not None else p

def predict_one(row: dict) -> dict:
    p = float(score_proba(pd.DataFrame([row]))[0])
    return {"p": p, "y": int(p >= THR), "thr": THR}

CONFIG["MODEL_BUNDLE_PATH"] = BUNDLE
print("Loaded:", BUNDLE)
print("Threshold:", THR, "| Features:", len(FEAT) if FEAT else "infer from df")

# Optional: log load event if your _append_event exists
try:
    _append_event({"type":"model_loaded","bundle":BUNDLE,"thr":THR,"n_features":(len(FEAT) if FEAT else None)})
except Exception:
    pass


Loaded: /kaggle/working/ed_phase2_model_thr_patched(1).joblib
Threshold: 0.9988444286248084 | Features: 16


In [17]:
# After the bridge cell
def score_and_log(row: dict):
    """row must have the 16 feature keys in the bundle (FEAT)."""
    res = predict_one(row)  # uses THR/FEAT from the bridge
    try:
        _append_event({"type":"ml_score", "res": res, "keys": list(row.keys())})
    except Exception:
        pass
    print(f"ML risk p={res['p']:.3f} → {'ALERT' if res['y'] else 'ok'} (thr={res['thr']:.3f})")
    return res


In [18]:
from joblib import load
b = load(CONFIG["MODEL_BUNDLE_PATH"])
print("thr:", b["threshold"], "n_features:", len(b["features"]))
# Dummy row shape-check:
print(set(b["features"]) - set((FEAT or [])))  # should be empty


thr: 0.9988444286248084 n_features: 16
set()


In [19]:

# Inline UI trigger (no sidecars)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W
        btn = W.Button(description="Run ML → OPS (inline)", button_style="primary")
        out = W.Output()
        def _go(_):
            with out:
                out.clear_output()
                print("Running…")
                try:
                    test, probs_te, tau, id_col, src = run_user_pipeline()
                    res = ml_to_ops_emit(test, probs_te, tau, id_col, src)
                    print("[OK] Emitted:", res)
                except Exception as e:
                    print("[ERROR]", e)
        btn.on_click(_go)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable:", e)
else:
    print("UI panel deferred… set CONFIG['RUN_UI']=True.")


In [20]:
# ICU availability panel — replaces "actionable vs blocked" with explicit next-bed ETAs.
# No sidecars; all inline; guarded by RUN_UI.
from pathlib import Path
from datetime import datetime, timezone
import json

# Pre-seeded UKE ICUs (public info): names + capacities
UKE_UNITS = [
    {"name": "1A Neurochirurgische Intensivstation", "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1B Neurologische Intensivstation",    "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1C Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1D Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1E Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1F Operative Intensivstation",        "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1G Internistische Intensivstation",   "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiologische Intensivstation",  "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiochirurgische Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H2b Intensivstation Gefäß- und Herzmedizin", "capacity": 8, "occupied": 8, "discharge_eta_minutes": []},
]

def _format_eta(mins: int) -> str:
    if mins is None: return "unknown"
    if mins <= 0: return "now"
    h, m = divmod(int(mins), 60)
    return f"{m} min" if h == 0 else (f"{h} hr" if m == 0 else f"{h} hr {m} min")

def _load_icu_status(path="/mnt/data/icu_status.json"):
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and js.get("units"):
                return js
        except Exception:
            pass
    # default to UKE units when nothing saved
    return {"units": list(UKE_UNITS), "timestamp": datetime.now(timezone.utc).isoformat()}

def _save_icu_status(js, path="/mnt/data/icu_status.json"):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(js, ensure_ascii=False, indent=2))

def _compute_next_bed_eta(unit):
    cap = int(unit.get("capacity", 0) or 0)
    occ = int(unit.get("occupied", 0) or 0)
    etas = [int(x) for x in (unit.get("discharge_eta_minutes") or []) if str(x).strip().isdigit()]
    if occ < cap: return 0
    return min(etas) if etas else None

if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W
        import pandas as pd

        state = _load_icu_status()
        units = state["units"]

        # UI widgets
        dd = W.Dropdown(options=[u.get("name","(unnamed)") for u in units] or ["(add a unit)"], description="Unit")
        name = W.Text(description="Name", placeholder="ICU-North")
        cap  = W.IntText(description="Capacity", value=12)
        occ  = W.IntText(description="Occupied", value=12)
        eta  = W.Text(description="ETAs (min)", placeholder="e.g. 30, 90, 180")

        add_btn = W.Button(description="Add/Update unit")
        calc_btn = W.Button(description="Estimate ETAs")
        save_btn = W.Button(description="Save status")
        out = W.Output()

        def _refresh_dropdown():
            dd.options = [u.get("name","(unnamed)") for u in units] or ["(add a unit)"]

        def _load_into_form(idx=0):
            if not units:
                name.value=""; cap.value=12; occ.value=12; eta.value=""; return
            u = units[idx]
            name.value = str(u.get("name",""))
            cap.value = int(u.get("capacity", 0) or 0)
            occ.value = int(u.get("occupied", 0) or 0)
            seq = u.get("discharge_eta_minutes") or []
            eta.value = ", ".join(str(int(x)) for x in seq)

        def _parse_eta(txt: str):
            out = []
            for chunk in txt.split(","):
                chunk = chunk.strip()
                if chunk:
                    try: out.append(int(float(chunk)))
                    except: pass
            return out

        def on_dd_change(change):
            if change["name"]=="value" and units:
                _load_into_form(dd.options.index(change["new"]))
        dd.observe(on_dd_change)

        def on_add(_):
            # no 'nonlocal' needed: we mutate the existing list
            u = {
                "name": name.value.strip() or f"ICU-{len(units)+1}",
                "capacity": int(cap.value or 0),
                "occupied": int(occ.value or 0),
                "discharge_eta_minutes": _parse_eta(eta.value),
            }
            names = [x.get("name","") for x in units]
            if u["name"] in names:
                units[names.index(u["name"])] = u
            else:
                units.append(u)
            _refresh_dropdown()
            dd.value = u["name"]
            with out:
                print(f"Saved unit '{u['name']}'")

        def on_calc(_):
            rows = []
            for u in units:
                eta_min = _compute_next_bed_eta(u)
                rows.append({
                    "ICU": u.get("name",""),
                    "capacity": int(u.get("capacity",0) or 0),
                    "occupied": int(u.get("occupied",0) or 0),
                    "next_bed_in": _format_eta(eta_min),
                })
            df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["ICU","capacity","occupied","next_bed_in"])
            with out:
                out.clear_output()
                if df.empty:
                    print("No units yet. Add a unit above.")
                else:
                    display(df.style.hide(axis='index'))
                    # Natural-language earliest
                    mins = [(r["ICU"], _compute_next_bed_eta(u)) for r,u in zip(rows, units)]
                    mins = [(n,m) for n,m in mins if m is not None]
                    if mins:
                        name_min, m = sorted(mins, key=lambda x: x[1])[0]
                        print(f"\nNext bed available on {name_min} in {_format_eta(m)}")
                    else:
                        print("\nNext bed availability: unknown (provide ETAs or reduce occupied < capacity).")

        def on_save(_):
            state["units"] = units
            state["timestamp"] = datetime.now(timezone.utc).isoformat()
            _save_icu_status(state)
            with out:
                print("Saved to /mnt/data/icu_status.json")

        add_btn.on_click(on_add)
        calc_btn.on_click(on_calc)
        save_btn.on_click(on_save)

        # Initial load
        _refresh_dropdown()
        if units:
            dd.value = dd.options[0]
            _load_into_form(0)

        display(W.VBox([
            W.HTML("<b>ICU next-bed availability</b>"),
            dd,
            W.HBox([name, cap, occ]),
            eta,
            W.HBox([add_btn, calc_btn, save_btn]),
            out
        ]))

    except Exception as e:
        print("ICU UI unavailable:", e)
else:
    print("ICU UI deferred… set CONFIG['RUN_UI']=True.")


In [21]:

# === Realistic synthetic data pipeline (inline, no sidecars) ===
# Guard: RUN_SYNTH controls generation + training + event emission
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd, numpy as np, json
from typing import Dict, Any, Optional
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve, precision_recall_fscore_support, roc_auc_score

BASE = Path("/mnt/data")
EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

def _detect_label_and_id(train: pd.DataFrame, meta_path=BASE/"meta.json"):
    meta = {}
    if Path(meta_path).exists():
        try:
            meta = json.loads(Path(meta_path).read_text())
        except Exception:
            meta = {}
    label_col = meta.get("label_col")
    if not label_col:
        for c in train.columns:
            if pd.api.types.is_numeric_dtype(train[c]):
                u = set(pd.unique(train[c].dropna()))
                if u.issubset({0,1}):
                    label_col = c; break
    if not label_col:
        raise RuntimeError("Could not detect label column")
    id_col = None
    for c in meta.get("validated_id_cols", []):
        if c in train.columns: id_col = c; break
    for c in ["Fall-ID","fall_id","PatientID","patient_id","VISIT_ID","visit_id","ID","id"]:
        if id_col is None and c in train.columns: id_col = c; break
    if id_col is None: id_col = train.columns[0]
    return label_col, id_col

def _split_features(df: pd.DataFrame, id_col: str, label_col: str):
    num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    cat = [c for c in df.columns if c not in num]
    drop = set([id_col, label_col])
    num = [c for c in num if c not in drop]
    cat = [c for c in cat if c not in drop]
    low_card = [c for c in cat if df[c].nunique(dropna=True) <= 30]
    return num, low_card

def _numeric_params(s: pd.Series):
    s_nonnull = s.dropna()
    if len(s_nonnull)==0:
        return {"mean":0.0,"std":1.0,"lo":0.0,"hi":1.0,"missing":1.0}
    mean = float(s_nonnull.mean()); std = float(s_nonnull.std(ddof=0) or 1.0)
    lo = float(np.percentile(s_nonnull, 1)); hi = float(np.percentile(s_nonnull, 99))
    missing = float(s.isna().mean())
    return {"mean":mean,"std":std,"lo":lo,"hi":hi,"missing":missing}

def _sample_numeric(n, p):
    x = np.random.normal(p["mean"], p["std"], size=n)
    x = np.clip(x, p["lo"], p["hi"])
    if p["missing"]>0:
        m = np.random.rand(n) < p["missing"]
        x = x.astype("float"); x[m] = np.nan
    return x

def _categorical_params(s: pd.Series):
    missing = float(s.isna().mean())
    counts = s.dropna().value_counts()
    if counts.empty:
        return {"cats":["UNK"],"probs":[1.0],"missing":1.0}
    cats = counts.index.tolist(); probs = (counts/counts.sum()).values.tolist()
    return {"cats":cats,"probs":probs,"missing":missing}

def _sample_categorical(n, p):
    base = np.random.choice(p["cats"], size=n, p=p["probs"])
    if p["missing"]>0:
        m = np.random.rand(n) < p["missing"]
        base = base.astype("object"); base[m] = None
    return base

def _prep_X(df: pd.DataFrame, num_cols, cat_cols, all_cols=None):
    X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, dummy_na=True)
    if all_cols is not None:
        for c in all_cols:
            if c not in X.columns: X[c] = 0
        X = X[all_cols]
    return X

def run_synth_pipeline(save_csv: bool=False):
    # Load real splits
    train = pd.read_csv(BASE/"train_DE_full.csv")
    val   = pd.read_csv(BASE/"val_DE_full.csv")
    test  = pd.read_csv(BASE/"test_DE_full.csv")

    label_col, id_col = _detect_label_and_id(train)
    num_cols, cat_cols = _split_features(train, id_col, label_col)

    # Fit a real model on real features (to induce structure for labels)
    Xr = _prep_X(train, num_cols, cat_cols); Xv = _prep_X(val, num_cols, cat_cols); Xt = _prep_X(test, num_cols, cat_cols)
    all_cols = sorted(set(Xr.columns) | set(Xv.columns) | set(Xt.columns))
    Xr = _prep_X(train, num_cols, cat_cols, all_cols); y_real = train[label_col].astype(int).values

    base_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler(with_mean=False)),
                          ("lr", LogisticRegression(max_iter=1000))])
    base_pipe.fit(Xr, y_real)
    target_prev = float(np.mean(y_real))

    # Params from real TRAIN for synthesis
    num_param_map = {c: _numeric_params(train[c]) for c in num_cols}
    cat_param_map = {c: _categorical_params(train[c]) for c in cat_cols}

    def _make_ids(n): 
        import uuid
        return [f"SYN-{uuid.uuid4().hex[:10]}" for _ in range(n)]

    def synth_df(n_rows: int) -> pd.DataFrame:
        data = {id_col: _make_ids(n_rows)}
        for c in cat_cols: data[c] = _sample_categorical(n_rows, cat_param_map[c])
        for c in num_cols: data[c] = _sample_numeric(n_rows, num_param_map[c])
        return pd.DataFrame(data)

    n_tr, n_va, n_te = len(train), len(val), len(test)
    syn_tr = synth_df(n_tr); syn_va = synth_df(n_va); syn_te = synth_df(n_te)

    def label_from_base(df: pd.DataFrame) -> np.ndarray:
        X = _prep_X(df, num_cols, cat_cols, all_cols)
        scores = base_pipe.predict_proba(X)[:,1]
        thr = np.quantile(scores, 1-target_prev) if 0<target_prev<1 else 0.5
        return (scores >= thr).astype(int)

    for df in (syn_tr, syn_va, syn_te):
        df[label_col] = label_from_base(df)

    # Reorder
    def _reorder(df): 
        cols = [id_col, label_col] + [c for c in df.columns if c not in [id_col, label_col]]
        return df[cols]
    syn_tr, syn_va, syn_te = map(_reorder, (syn_tr, syn_va, syn_te))

    # Train calibrated model on SYNTH
    X_tr = _prep_X(syn_tr, num_cols, cat_cols); y_tr = syn_tr[label_col].astype(int).values
    X_va = _prep_X(syn_va, num_cols, cat_cols); y_va = syn_va[label_col].astype(int).values
    X_te = _prep_X(syn_te, num_cols, cat_cols); y_te = syn_te[label_col].astype(int).values
    all_synth_cols = sorted(set(X_tr.columns) | set(X_va.columns) | set(X_te.columns))
    X_tr = _prep_X(syn_tr, num_cols, cat_cols, all_synth_cols)
    X_va = _prep_X(syn_va, num_cols, cat_cols, all_synth_cols)
    X_te = _prep_X(syn_te, num_cols, cat_cols, all_synth_cols)

    base = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc", StandardScaler(with_mean=False)),
                     ("lr", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
    cal = CalibratedClassifierCV(base, method="isotonic", cv="prefit").fit(X_va, y_va)

    probs_va = cal.predict_proba(X_va)[:,1]
    fpr, tpr, thr = roc_curve(y_va, probs_va)
    RECALL_FLOOR = 0.85
    meet = np.where(tpr >= RECALL_FLOOR)[0]
    if len(meet)>0:
        tau = float(thr[meet[0]])
    else:
        best_f1, best_t = -1, 0.5
        for t in np.linspace(0,1,201):
            yb = (probs_va >= t).astype(int)
            _, _, f1, _ = precision_recall_fscore_support(y_va, yb, average="binary", zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        tau = float(best_t)

    probs_te = cal.predict_proba(X_te)[:,1]
    auc_te = float(roc_auc_score(y_te, probs_te))

    # Emit events (ml_risk_synth + lingering_alert_synth + summary)
    n_events = 0; n_pos = 0
    for i in range(len(syn_te)):
        pid = syn_te.iloc[i][id_col]
        p = float(probs_te[i])
        decision = "POS" if p >= tau else "NEG"
        ev = {"type":"ml_risk_synth","patient_id": pid,"id_col": id_col,
              "prob_cal": round(p,6), "tau": round(tau,6),"decision": decision,
              "source":"synth_pipeline"}
        _append_event(ev)
        n_events += 1
        if decision=="POS":
            n_pos += 1
            _append_event({"type":"lingering_alert_synth","patient_id": pid,"id_col": id_col,
                           "prob_cal": round(p,6), "tau": round(tau,6),
                           "reason":"ml_high_risk_synth","source":"LingeringPatientMonitor"})

    _append_event({"type":"ml_risk_summary_synth","tau": round(tau,6),"n_rows": int(len(syn_te)),
                   "id_col": id_col, "auc_te": round(auc_te,6), "source":"synth_pipeline"})

    # Optionally save CSVs
    if CONFIG.get("SAVE_SYNTH"):
        syn_tr.to_csv(BASE/"synth_train.csv", index=False)
        syn_va.to_csv(BASE/"synth_val.csv", index=False)
        syn_te.to_csv(BASE/"synth_test.csv", index=False)

    return {"n_events": n_events, "n_pos": n_pos, "tau": tau, "auc_te": auc_te, "id_col": id_col}

if CONFIG.get("RUN_SYNTH"):
    try:
        res = run_synth_pipeline(save_csv=bool(CONFIG.get("SAVE_SYNTH", False)))
        print("[SYNTH OK]", res)
        print("Events in:", CONFIG["EVENT_LOG_PATH"])
    except Exception as e:
        print("[SYNTH ERROR]", e)
else:
    print("Deferred… set CONFIG['RUN_SYNTH']=True to generate + run synthetic pipeline.")


Deferred… set CONFIG['RUN_SYNTH']=True to generate + run synthetic pipeline.


In [22]:
# Extend CONFIG for meds integration (idempotent, Kaggle-safe)
CONFIG.setdefault("MED_RULES_PATH", _p("interaction_rules.json"))
CONFIG.setdefault("ALLERGIES_PATH", _p("patient_allergies.json"))

from pathlib import Path
import json, os

# Ensure files exist and are valid JSON arrays (no seeding of drug examples here)
for k in ("MED_RULES_PATH","ALLERGIES_PATH"):
    p = Path(CONFIG[k]); p.parent.mkdir(parents=True, exist_ok=True)
    if not p.exists() or os.path.getsize(p) == 0:
        p.write_text("[]", encoding="utf-8")
    try:
        json.loads(p.read_text(encoding="utf-8") or "[]")
    except Exception:
        p.write_text("[]", encoding="utf-8")
print("Meds/allergy paths →", CONFIG["MED_RULES_PATH"], "|", CONFIG["ALLERGIES_PATH"])


Meds/allergy paths → /mnt/data/interaction_rules.json | /mnt/data/patient_allergies.json


In [23]:
# === Phase-2 calculators (extended, patched) + bundle + HL7 paste adapter ===
# Add this cell beneath your CONFIG/guards. No sidecars. Kaggle/Colab/Local safe.

from __future__ import annotations
from datetime import datetime, timezone
from typing import Dict, Any, Optional, List, Tuple
import json, re, math

# ---- Event sink wiring (uses your CONFIG if present) ----
try:
    EVENT_LOG_PATH = CONFIG.get("EVENT_LOG_PATH", "/mnt/data/event_log.jsonl")
except NameError:
    CONFIG = {"RUN_PIPELINE": True, "RUN_UI": True, "EVENT_LOG_PATH": "/mnt/data/event_log.jsonl"}
    EVENT_LOG_PATH = CONFIG["EVENT_LOG_PATH"]
    print("[guards] CONFIG was missing — seeded defaults:", CONFIG)

def _append_event(ev: Dict[str, Any]):
    ev = {"ts": datetime.utcnow().isoformat()+"Z", **ev}
    with open(EVENT_LOG_PATH, "a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# ---- helpers ----
def _pick(d: Dict[str, Any], names: List[str], cast=float, default=None):
    for n in names:
        if n in d and d[n] not in (None, ""):
            try: return cast(d[n])
            except Exception:
                try: return cast(str(d[n]).replace(",", "."))
                except Exception: pass
    return default
def _bool(v): 
    if isinstance(v, bool): return v
    s = str(v).strip().lower()
    return s in {"1","true","yes","y","ja","on","oui"}
def _safe_round(x, nd=1):
    try: return round(float(x), nd)
    except Exception: return x

# ---- calculators ----
def calc_qsofa(v: Dict[str,Any]): 
    rr=_pick(v,["rr","resp_rate"]); sbp=_pick(v,["sbp","systolic"]); gcs=_pick(v,["gcs"],float)
    avpu = (v.get("avpu") or "").upper()[:1]
    altered = (gcs is not None and gcs<15) or avpu in {"V","P","U"}
    return {"name":"qSOFA","score": int((rr is not None and rr>=22)) + int((sbp is not None and sbp<=100)) + int(altered)}

def calc_mews(v: Dict[str,Any]):
    def rr_s(x):  return 3 if x is not None and x<=8 else (0 if x and 9<=x<=14 else (1 if x and 15<=x<=20 else (2 if x and 21<=x<=29 else (3 if x and x>=30 else 0))))
    def hr_s(x):  return 2 if x is not None and x<=40 else (1 if x and 41<=x<=50 else (0 if x and 51<=x<=100 else (1 if x and 101<=x<=110 else (2 if x and 111<=x<=129 else (3 if x and x>=130 else 0)))))
    def sbp_s(x): return 3 if x is not None and x<=70 else (2 if x and 71<=x<=80 else (1 if x and 81<=x<=100 else (0 if x and 101<=x<=199 else (2 if x and x>=200 else 0))))
    def t_s(x):   return 2 if x is not None and x<=35.0 else (1 if x and 35.1<=x<=36.0 else (0 if x and 36.1<=x<=38.0 else (1 if x and 38.1<=x<=38.5 else (2 if x and x>=38.6 else 0))))
    def avpu_s(x): return {"A":0,"V":1,"P":2,"U":3}.get((x or "A").upper()[:1],0)
    return {"name":"MEWS","score": int(rr_s(_pick(v,["rr"]))+hr_s(_pick(v,["hr","pulse"]))+sbp_s(_pick(v,["sbp"]))+t_s(_pick(v,["temp"]))+avpu_s(v.get("avpu")))}

def calc_heart(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hist=_pick(p,["heart_history"],int); ecg=_pick(p,["heart_ecg"],int); risk=_pick(p,["heart_risk"],int)
    trop=_pick(p,["troponin","hs_troponin","trop"]); uln=_pick(p,["troponin_uln"],float)
    age_s = 2 if (age is not None and age>=65) else (1 if (age is not None and 45<=age<=64) else 0)
    ratio = (trop/uln) if (trop is not None and uln) else None
    trop_s = 2 if (ratio is not None and ratio>3) else (1 if (ratio is not None and 1<ratio<=3) else (0 if ratio is not None else 0))
    total = (hist or 0)+(ecg or 0)+age_s+(risk or 0)+trop_s
    return {"name":"HEART","score": int(total)}

def calc_grace_coarse(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hr=_pick(p,["hr","pulse"]); sbp=_pick(p,["sbp"]); crea=_pick(p,["creatinine"])
    sc=0
    if age is not None: sc += (0 if age<40 else 20 if age<60 else 40 if age<80 else 60)
    if hr  is not None: sc += (0 if hr<70  else 10 if hr<90  else 20 if hr<110 else 30 if hr<150 else 40)
    if sbp is not None: sc += (40 if sbp<80 else 30 if sbp<100 else 10 if sbp<120 else 0)
    if crea is not None: sc += (0 if crea<1.2 else 10 if crea<2.0 else 20 if crea<3.0 else 30)
    return {"name":"GRACE_coarse","score": int(sc)}

def calc_sofa_min(v: Dict[str,Any], labs: Dict[str,Any]):
    pf=None; pao2=_pick(labs,["pao2"]); fio2=_pick(labs,["fio2"])
    if pao2 is not None and fio2: 
        try: pf=float(pao2)/float(fio2)
        except Exception: pf=None
    plate=_pick(labs,["platelets","plt"]); bili=_pick(labs,["bilirubin"]); mapv=_pick(v,["map"]); gcs=_pick(v,["gcs"]); crea=_pick(labs,["creatinine"])
    sc=0
    if pf is not None:    sc += (4 if pf<100 else 3 if pf<200 else 2 if pf<300 else 1 if pf<400 else 0)
    if plate is not None: sc += (4 if plate<20 else 3 if plate<50 else 2 if plate<100 else 1 if plate<150 else 0)
    if bili  is not None: sc += (4 if bili>=12 else 3 if bili>=6 else 2 if bili>=2 else 1 if bili>=1.2 else 0)
    if mapv  is not None: sc += (1 if mapv<70 else 0)
    if gcs   is not None: sc += (4 if gcs<6 else 3 if gcs<10 else 2 if gcs<13 else 1 if gcs<15 else 0)
    if crea  is not None: sc += (4 if crea>=5 else 3 if crea>=3.5 else 2 if crea>=2 else 1 if crea>=1.2 else 0)
    return {"name":"SOFA_min","score": int(sc)}

def assess_d_dimer(value, unit, age, pregnant=False):
    if value is None: return {"available": False}
    unit=(unit or "").lower()
    val_ug = float(value)*1000.0 if "mg/l" in unit else float(value)  # assume μg/L FEU default
    thr = 500.0
    if age is not None and age>50 and not pregnant: thr = float(age)*10.0
    return {"available": True, "value_ug_per_l": val_ug, "thr_ug_per_l": thr, "ok_below_thr": bool(val_ug < thr)}

def assess_troponin_delta(series: List[Tuple[Optional[datetime], float, str]]):
    if not series or len(series)<2: return {"available": False}
    try: series = sorted(series, key=lambda x: (x[0] or datetime.min))
    except Exception: pass
    (t0,v0,u0),(t1,v1,u1)=series[-2],series[-1]
    def to_ng_l(v,u):
        u=(u or "").lower()
        if "ng/l" in u or "pg/ml" in u: return float(v)
        if "µg/l" in u or "ug/l" in u:  return float(v)*1000.0
        return float(v)
    p=to_ng_l(v0,u0); c=to_ng_l(v1,u1); d=c-p; pct=(abs(d)/p*100.0) if p!=0 else None
    flag = (abs(d)>=51.0) or (pct is not None and pct>=20.0)
    return {"available": True, "prev": p, "curr": c, "delta_abs": d, "delta_pct": pct, "flag": bool(flag)}

def calc_wells_pe(p: Dict[str,Any]):
    pts=0.0
    pts+=3.0 if _bool(p.get("dvt_signs")) else 0.0
    pts+=3.0 if _bool(p.get("pe_most_likely")) else 0.0
    pts+=1.5 if ((_pick(p,["hr","pulse"],float) or 0)>100) else 0.0
    pts+=1.5 if (_bool(p.get("immobilized")) or _bool(p.get("recent_surgery_4w"))) else 0.0
    pts+=1.5 if _bool(p.get("prev_vte")) else 0.0
    pts+=1.0 if _bool(p.get("hemoptysis")) else 0.0
    pts+=1.0 if _bool(p.get("cancer_active")) else 0.0
    return {"name":"WELLS_PE","score": pts, "tier2": ("likely" if pts>4 else "unlikely")}

def calc_wells_dvt(p: Dict[str,Any]):
    pts=0
    pts+=1 if _bool(p.get("cancer_active")) else 0
    pts+=1 if (_bool(p.get("paresis")) or _bool(p.get("plaster_cast"))) else 0
    pts+=1 if (_bool(p.get("bedridden_3d")) or _bool(p.get("surgery_12w"))) else 0
    pts+=1 if _bool(p.get("deep_vein_tenderness")) else 0
    pts+=1 if _bool(p.get("entire_leg_swollen")) else 0
    pts+=1 if _bool(p.get("calf_swelling_gt3cm")) else 0
    pts+=1 if _bool(p.get("pitting_edema")) else 0
    pts+=1 if _bool(p.get("collateral_nonvaricose")) else 0
    pts+=1 if _bool(p.get("prev_dvt")) else 0
    pts-=2 if _bool(p.get("alt_dx_as_likely")) else 0
    return {"name":"WELLS_DVT","score": int(pts), "tier2": ("likely" if pts>=2 else "unlikely")}

def calc_perc(p: Dict[str,Any]):
    crit = {
        "age<50": int((_pick(p,["age"],int) or 10) < 50),
        "hr<100": int((_pick(p,["hr","pulse"],float) or 0) < 100),
        "sao2>=95": int((_pick(p,["sao2","spo2"],float) or 0) >= 95),
        "no_hemoptysis": int(not _bool(p.get("hemoptysis"))),
        "no_estrogen": int(not _bool(p.get("estrogen_use"))),
        "no_surg/trauma_4w": int(not (_bool(p.get("recent_surgery_4w")) or _bool(p.get("recent_trauma_4w")))),
        "no_prior_vte": int(not _bool(p.get("prev_vte"))),
        "no_unilateral_swelling": int(not _bool(p.get("unilateral_leg_swelling"))),
    }
    return {"name":"PERC","passed": bool(all(crit.values()))}

def calc_pesi(p: Dict[str,Any]):
    age=_pick(p,["age"],int) or 0
    male = 10 if (str(p.get("sex") or "").upper().startswith("M")) else 0
    cancer = 30 if _bool(p.get("cancer_active")) else 0
    hf = 10 if _bool(p.get("heart_failure")) else 0
    lung = 10 if (_bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))) else 0
    hr = 20 if ((_pick(p,["hr","pulse"],float) or 0) >=110) else 0
    sbp = 30 if ((_pick(p,["sbp"],float) or 200) < 100) else 0
    rr = 20 if ((_pick(p,["rr"],float) or 0) >=30) else 0
    temp = 20 if ((_pick(p,["temp"],float) or 37) < 36) else 0
    altered = 60 if ((_pick(p,["gcs"],float) or 15) < 15) else 0
    sat = 20 if ((_pick(p,["sao2","spo2"],float) or 100) < 90) else 0
    score = age+male+cancer+hf+lung+hr+sbp+rr+temp+altered+sat
    klass = "I" if score<=65 else "II" if score<=85 else "III" if score<=105 else "IV" if score<=125 else "V"
    return {"name":"PESI","score": int(score), "class": klass}

def calc_spesi(p: Dict[str,Any]):
    comps = {
        "age>80": int((_pick(p,["age"],int) or 0) > 80),
        "cancer": int(_bool(p.get("cancer_active"))),
        "cardiopulm": int(_bool(p.get("heart_failure")) or _bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))),
        "hr>=110": int((_pick(p,["hr","pulse"],float) or 0) >= 110),
        "sbp<100": int((_pick(p,["sbp"],float) or 200) < 100),
        "o2<90": int((_pick(p,["sao2","spo2"],float) or 100) < 90),
    }
    return {"name":"sPESI","score": int(sum(comps.values()))}

def calc_sirs(p: Dict[str,Any]):
    crit = {
        "temp>38/<36": int(((_pick(p,["temp"],float) or 37)>38) or ((_pick(p,["temp"],float) or 37)<36)),
        "hr>90": int((_pick(p,["hr","pulse"],float) or 0) > 90),
        "rr>20/paco2<32": int(((_pick(p,["rr"],float) or 0) > 20) or ((_pick(p,["paco2"],float) or 100) < 32)),
        "wbc>12/<4/bands>10%": int(((_pick(p,["wbc"],float) or 7) > 12) or ((_pick(p,["wbc"],float) or 7) < 4) or ((_pick(p,["bands_pct"],float) or 0) > 10)),
    }
    return {"name":"SIRS","score": int(sum(crit.values()))}

def sepsis3_screen(v: Dict[str,Any], labs: Dict[str,Any], ctx: Dict[str,Any], sofa_min: Dict[str,Any], qsofa: Dict[str,Any]):
    sus = _bool(ctx.get("suspected_infection")); on_pressors=_bool(ctx.get("vasopressors"))
    mapv=_pick(v,["map"]); lact=_pick(labs,["lactate"])
    septic_shock = bool((mapv is not None and mapv<65) and (lact is not None and lact>2) and on_pressors)
    return {"name":"SEPSIS3","sepsis_flag": bool(sus and sofa_min["score"]>=2), "septic_shock": septic_shock}

def calc_marburg(p: Dict[str,Any]):
    sex=(str(p.get("sex") or "")[:1]).upper(); age=_pick(p,["age"],int)
    vasc=_bool(p.get("vasc_disease")); exert=_bool(p.get("exertional")); pt_thinks=_bool(p.get("patient_assumes_cardiac"))
    palp = p.get("palpation_reproducible"); not_repro = (palp is False)
    age_sex = ((sex=="M" and age is not None and age>=55) or (sex=="F" and age is not None and age>=65))
    sc = int(bool(age_sex)) + int(vasc) + int(exert) + int(pt_thinks) + int(bool(not_repro))
    return {"name":"MARBURG","score": int(sc)}

def calc_gbs(p: Dict[str,Any]):
    score=0
    urea=_pick(p,["urea_mmol_l","urea"]); bun=_pick(p,["bun_mg_dl"])
    if urea is None and bun is not None: urea=float(bun)/2.8
    hb_gL=_pick(p,["hb_g_l"]); 
    if hb_gL is None:
        hb_gdl=_pick(p,["hb","hb_g_dl"]); 
        if hb_gdl is not None: hb_gL=hb_gdl*10.0
    sbp=_pick(p,["sbp"]); hr=_pick(p,["hr","pulse"]); male = str(p.get("sex") or "").upper().startswith("M")
    if urea is not None:
        score += 2 if 6.5<=urea<=7.9 else 0; score += 3 if 8.0<=urea<=9.9 else 0
        score += 4 if 10.0<=urea<=25.0 else 0; score += 6 if urea>25.0 else 0
    if hb_gL is not None:
        if male:   score += (1 if 120<=hb_gL<=129 else 0) + (3 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
        else:      score += (1 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
    if sbp is not None: score += (1 if 100<=sbp<=109 else 0) + (2 if 90<=sbp<=99 else 0) + (3 if sbp<90 else 0)
    score += 1 if (hr is not None and hr>=100) else 0
    score += 1 if _bool(p.get("melena")) else 0
    score += 2 if _bool(p.get("syncope")) else 0
    score += 2 if _bool(p.get("hepatic_disease")) else 0
    score += 2 if _bool(p.get("cardiac_failure")) else 0
    return {"name":"GBS","score": int(score)}

def calc_child_pugh(p: Dict[str,Any]):
    bili=_pick(p,["bilirubin"]); alb=_pick(p,["albumin"]); inr=_pick(p,["inr"])
    asc=(p.get("ascites") or "").lower(); ence=(p.get("encephalopathy") or "").lower()
    sc=0; filled=0
    if bili is not None: sc+=(1 if bili<2 else 2 if bili<=3 else 3); filled+=1
    if alb  is not None: sc+=(1 if alb>3.5 else 2 if alb>=2.8 else 3); filled+=1
    if inr  is not None: sc+=(1 if inr<1.7 else 2 if inr<=2.3 else 3); filled+=1
    if asc:              sc+=(1 if asc.startswith("n") else 2 if asc.startswith(("mild","slight")) else 3); filled+=1
    if ence:             sc+=(1 if ence in {"none","0"} else 2 if any(x in ence for x in ["1","2","i","ii"]) else 3); filled+=1
    if filled<5: return {"name":"CHILD_PUGH","score_partial": int(sc), "class":"incomplete"}
    klass = "A" if sc<=6 else ("B" if sc<=9 else "C")
    return {"name":"CHILD_PUGH","score": int(sc), "class": klass}

def corrected_calcium(total_ca, albumin, units="mg/dL", normal_alb=None):
    if total_ca is None or albumin is None: return None
    if "mmol" in (units or "").lower():
        alb_gl = albumin if albumin>10 else albumin*10.0
        normal = 40.0 if normal_alb is None else float(normal_alb)
        return float(total_ca) + 0.02*(normal - alb_gl)
    normal = 4.0 if normal_alb is None else float(normal_alb)
    return float(total_ca) + 0.8*(normal - float(albumin))

def anion_gap(na, cl, hco3, k=None, albumin_gdl=None):
    if na is None or cl is None or hco3 is None: return None
    ag = (float(na)+(float(k) if k is not None else 0.0)) - (float(cl)+float(hco3))
    agc = ag + (2.5*(4.0 - float(albumin_gdl))) if albumin_gdl is not None else None
    return {"ag": _safe_round(ag,1), "ag_albumin_corrected": _safe_round(agc,1) if agc is not None else None}

# ---- HL7 (Troponin/D-dimer + INR/Bili/Albumin) ----
def parse_hl7_labs(hl7_text: str) -> Dict[str, Any]:
    troponin, ddimer = [], []
    others = {"bilirubin": None, "inr": None, "albumin": None}
    lines = re.split(r'[\r\n]+', (hl7_text or "").strip())
    for ln in lines:
        if not ln.strip(): continue
        parts = ln.split("|")
        obx3 = parts[3] if len(parts)>3 else ""
        obx5 = parts[5] if len(parts)>5 else ""
        obx6 = parts[6] if len(parts)>6 else ""
        obx14= parts[14] if len(parts)>14 else ""
        name=(obx3 or ln); val=obx5; unit=obx6
        ts=None
        mdt = re.search(r'(\d{8})(\d{6})?', obx14)
        if mdt:
            try: ts = datetime.strptime(mdt.group(1)+(mdt.group(2) or "000000"), "%Y%m%d%H%M%S")
            except Exception: ts=None
        try: v=float(str(val).replace(",", "."))
        except Exception:
            mnum=re.search(r'[-+]?\d*\.?\d+', str(val).replace(",", "."))
            v=float(mnum.group(0)) if mnum else None
        if not unit:
            munit=re.search(r'\[(.*?)\]', name)
            if munit: unit=munit.group(1)
        n=name.upper()
        if ("TROP" in n or "TROPONIN" in n) and v is not None: troponin.append((ts,v,unit or "ng/L"))
        if ("D-DIMER" in n or "DDIMER" in n or "D DIMER" in n) and v is not None: ddimer.append((ts,v,unit or "μg/L FEU"))
        if v is not None:
            if ("BILIRUBIN" in n or "BILI" in n) and others["bilirubin"] is None:
                others["bilirubin"] = (float(v)/17.1) if (unit or "").lower() in {"umol/l","µmol/l"} else float(v)
            if "INR" in n and others["inr"] is None: others["inr"]=float(v)
            if "ALBUMIN" in n and others["albumin"] is None:
                a=float(v)
                others["albumin"] = a/10.0 if (unit or "").lower() in {"g/l","g l","gl"} else a
    return {"troponin": troponin, "d_dimer": ddimer, "others": others}

# ---- Bundle / one-liners ----
def phase2_bundle(vitals: Dict[str,Any], labs: Dict[str,Any], context: Optional[Dict[str,Any]]=None) -> Dict[str,Any]:
    context=context or {}
    age=_pick({**vitals, **context},["age"],int)
    pregnant=bool(context.get("pregnant", False))

    s_qsofa=calc_qsofa(vitals)
    s_mews =calc_mews(vitals)
    s_heart=calc_heart({**vitals, **labs})
    s_grace=calc_grace_coarse({**vitals, **labs})
    s_sofa =calc_sofa_min(vitals, labs)

    d_val=_pick(labs,["d_dimer","ddimer","d-dimer","d_dimer_feu_ug_l"]); d_unit=labs.get("d_dimer_unit") or "μg/L FEU"
    d_assess = assess_d_dimer(d_val, d_unit, age, pregnant) if d_val is not None else {"available": False}

    series=[]
    for it in labs.get("troponin_series") or []:
        if isinstance(it, dict):
            ts=None
            if it.get("time"):
                try: ts=datetime.fromisoformat(str(it["time"]).replace("Z",""))
                except Exception: ts=None
            series.append((ts, _pick(it,["value","val"]), it.get("unit","ng/L")))
    t_assess = assess_troponin_delta(series) if series else {"available": False}

    wells_pe  = calc_wells_pe({**vitals, **labs, **context})
    wells_dvt = calc_wells_dvt({**vitals, **labs, **context})
    perc      = calc_perc({**vitals, **labs, **context})
    pesi      = calc_pesi({**vitals, **labs, **context})
    spesi     = calc_spesi({**vitals, **labs, **context})
    sirs      = calc_sirs({**vitals, **labs})
    sepsis3   = sepsis3_screen(vitals, labs, context, s_sofa, s_qsofa)
    marburg   = calc_marburg({**vitals, **labs, **context})
    gbs       = calc_gbs({**vitals, **labs, **context})
    childpugh = calc_child_pugh({**vitals, **labs, **context})
    ca_corr   = corrected_calcium(_pick(labs,["calcium","ca","calcium_mg_dl"]), _pick(labs,["albumin","alb","albumin_g_dl"]), units="mg/dL")
    ag_val    = anion_gap(_pick(labs,["na","sodium"]), _pick(labs,["cl","chloride"]), _pick(labs,["hco3","bicarbonate"]),
                          k=_pick(labs,["k","potassium"]), albumin_gdl=_pick(labs,["albumin","alb","albumin_g_dl"]))

    ones=[]
    ones.append(f"qSOFA={s_qsofa['score']}  MEWS={s_mews['score']}  GRACE(coarse)={s_grace['score']}")
    ones.append(f"HEART={s_heart['score']}  SOFA(min)={s_sofa['score']}")
    if d_assess.get("available"):
        ones.append(f"D-dimer {int(d_assess['value_ug_per_l'])} vs thr {int(d_assess['thr_ug_per_l'])} μg/L → {'OK' if d_assess['ok_below_thr'] else 'High'}")
    if t_assess.get("available"):
        da=_safe_round(t_assess['delta_abs'],1); dp=_safe_round(t_assess['delta_pct'],1) if t_assess['delta_pct'] is not None else None
        ones.append(f"Troponin Δ {da} ng/L ({dp}%) → {'FLAG' if t_assess['flag'] else 'ok'}")
    ones.append(f"Wells-PE={_safe_round(wells_pe['score'],1)} ({wells_pe['tier2']})  Wells-DVT={wells_dvt['score']} ({wells_dvt['tier2']})")
    ones.append(f"PERC={'pass' if perc['passed'] else 'fail'}  sPESI={spesi['score']}  PESI={pesi['class']}/{pesi['score']}")
    ones.append(f"SIRS={sirs['score']}  Sepsis3: {'YES' if sepsis3['sepsis_flag'] else 'no'}  Shock: {'YES' if sepsis3['septic_shock'] else 'no'}")
    ones.append(f"MARBURG={marburg['score']}/5  GBS={gbs['score']}")
    if childpugh.get("class") == "incomplete":
        ones.append("Child-Pugh incomplete (need 5/5 inputs)")
    else:
        ones.append(f"Child-Pugh {childpugh['class']} ({childpugh['score']})")
    if ca_corr is not None: ones.append(f"Corrected Ca={_safe_round(ca_corr,2)} mg/dL")
    if ag_val is not None:
        ab = f", AGcorr={ag_val['ag_albumin_corrected']}" if ag_val.get("ag_albumin_corrected") is not None else ""
        ones.append(f"Anion gap={ag_val['ag']}{ab}")

    out = {
        "one_liners": ones,
        "scores": {
            "qsofa": s_qsofa["score"], "mews": s_mews["score"], "heart": s_heart["score"], "grace_coarse": s_grace["score"],
            "sofa_min": s_sofa["score"], "wells_pe": wells_pe["score"], "wells_dvt": wells_dvt["score"],
            "pesi": pesi["score"], "spesi": spesi["score"], "sirs": sirs["score"], 
            "sepsis3": int(sepsis3["sepsis_flag"]), "gbs": gbs["score"], "marburg": marburg["score"],
            "child_pugh": childpugh.get("score") or childpugh.get("score_partial")
        },
        "labs": {
            "d_dimer": d_val, "d_dimer_thr": d_assess.get("thr_ug_per_l"),
            "trop_prev": t_assess.get("prev"), "trop_curr": t_assess.get("curr"),
            "trop_delta": t_assess.get("delta_abs"), "trop_delta_pct": t_assess.get("delta_pct"), "trop_flag": t_assess.get("flag")
        }
    }
    return out



In [24]:
# ---- UI (ipywidgets) + HL7 merge ----
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd, json
        EVENT_LOG_PATH = CONFIG.get("EVENT_LOG_PATH", "/mnt/data/event_log.jsonl")

        # --- Text areas ---
        vitals_in = W.Textarea(
            value='{"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97}',
            description="Vitals JSON", layout=W.Layout(width="100%", height="90px")
        )
        labs_in = W.Textarea(
            value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU","troponin_series":[{"time":"2025-08-20T05:00:00Z","value":18,"unit":"ng/L"},{"time":"2025-08-20T09:00:00Z","value":78,"unit":"ng/L"}],"creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20,"bilirubin":2.1,"inr":1.9}',
            description="Labs JSON", layout=W.Layout(width="100%", height="130px")
        )
        ctx_in = W.Textarea(
            value='{"pregnant": false, "suspected_infection": true, "vasopressors": false, "pe_most_likely": true, "dvt_signs": false, "ascites":"mild","encephalopathy":"1-2"}',
            description="Context JSON", layout=W.Layout(width="100%", height="90px")
        )

        # --- HEART component inputs (0–2) ---
        heart_hist = W.Dropdown(options=[0,1,2], value=0, description="HEART: History")
        heart_ecg  = W.Dropdown(options=[0,1,2], value=0, description="HEART: ECG")
        heart_risk = W.Dropdown(options=[0,1,2], value=0, description="HEART: Risk")

        # --- PE / PERC flags ---
        chk_pe_most  = W.Checkbox(description="PE most likely", value=True)
        chk_dvt_signs= W.Checkbox(description="DVT signs", value=False)
        chk_immob    = W.Checkbox(description="Immobilized / recent surgery (4w)", value=False)
        chk_prev_vte = W.Checkbox(description="Prior VTE", value=False)
        chk_hemo     = W.Checkbox(description="Hemoptysis", value=False)
        chk_cancer   = W.Checkbox(description="Active malignancy", value=False)
        chk_estrogen = W.Checkbox(description="Estrogen use (PERC)", value=False)
        chk_trauma   = W.Checkbox(description="Recent trauma (4w, PERC)", value=False)
        chk_unilat   = W.Checkbox(description="Unilateral leg swelling (PERC)", value=False)

        # --- Wells-DVT flags ---
        dvt_cancer   = W.Checkbox(description="Active cancer", value=False)
        dvt_paresis  = W.Checkbox(description="Paresis / plaster cast", value=False)
        dvt_bed_surg = W.Checkbox(description="Bedridden ≥3d / surgery ≤12w", value=False)
        dvt_tender   = W.Checkbox(description="Deep vein tenderness", value=False)
        dvt_entire   = W.Checkbox(description="Entire leg swollen", value=False)
        dvt_calf3    = W.Checkbox(description="Calf swelling >3 cm", value=False)
        dvt_edema    = W.Checkbox(description="Pitting edema (symptomatic leg)", value=False)
        dvt_collat   = W.Checkbox(description="Collateral non-varicose", value=False)
        dvt_prev     = W.Checkbox(description="Previous DVT", value=False)
        dvt_alt_dx   = W.Checkbox(description="Alternative dx as likely (subtract)", value=False)

        # --- HL7 paste area ---
        hl7_in = W.Textarea(
            value='''OBX|1|NM|BILIRUBIN TOTAL||36|umol/L|||N||F|||20250820090000
OBX|2|NM|INR||1.9|||N||F|||20250820090000
OBX|3|NM|ALBUMIN||28|g/L|||N||F|||20250820090000
OBX|4|NM|D-DIMER||780|ug/L|||H||F|||20250820090000
OBX|5|NM|TROPONIN T||78|ng/L|||H||F|||20250820090000
OBX|6|NM|TROPONIN T||18|ng/L|||N||F|||20250820050000''',
            description="HL7 paste", layout=W.Layout(width="100%", height="140px")
        )

        btn_apply_hl7 = W.Button(description="Apply HL7 → Labs JSON")
        btn_compute   = W.Button(description="Compute + Log")
        out = W.Output()

        def on_apply(_):
            with out:
                out.clear_output()
                try:
                    labs = json.loads(labs_in.value or "{}")
                except Exception as e:
                    print("[labs parse error]", e)
                    return
                parsed = parse_hl7_labs(hl7_in.value or "")
                # Merge core others
                oth = parsed.get("others") or {}
                for k in ["bilirubin","inr","albumin"]:
                    if oth.get(k) is not None:
                        labs[k] = oth[k]
                # If available, set D-dimer (take newest) when missing
                dd = parsed.get("d_dimer") or []
                if dd and "d_dimer" not in labs:
                    dd_sorted = sorted(dd, key=lambda t: (t[0] or 0))
                    _, v, u = dd_sorted[-1]
                    labs["d_dimer"] = v
                    labs["d_dimer_unit"] = u
                # Always (re)write troponin_series if present
                ts = parsed.get("troponin") or []
                if ts:
                    series = []
                    for t, v, u in ts:
                        series.append({
                            "time": (t.isoformat()+"Z") if t else None,
                            "value": v, "unit": u or "ng/L"
                        })
                    labs["troponin_series"] = series
                labs_in.value = json.dumps(labs, ensure_ascii=False)
                print("HL7 merged →", {k: labs.get(k) for k in ["bilirubin","inr","albumin","d_dimer","d_dimer_unit"]})

        def on_compute(_):
            with out:
                out.clear_output()
                try:
                    vitals = json.loads(vitals_in.value or "{}")
                    labs   = json.loads(labs_in.value or "{}")
                    ctx    = json.loads(ctx_in.value or "{}")
                except Exception as e:
                    print("[parse error]", e)
                    return

                # HEART sub-scores from dropdowns
                vitals["heart_history"] = int(heart_hist.value)
                vitals["heart_ecg"]     = int(heart_ecg.value)
                vitals["heart_risk"]    = int(heart_risk.value)

                # PE / PERC + DVT flags
                ctx.update({
                    "pe_most_likely": chk_pe_most.value,
                    "dvt_signs": chk_dvt_signs.value,
                    "immobilized": chk_immob.value,
                    "recent_surgery_4w": chk_immob.value,
                    "prev_vte": chk_prev_vte.value,
                    "hemoptysis": chk_hemo.value,
                    "cancer_active": chk_cancer.value or ctx.get("cancer_active", False),
                    "estrogen_use": chk_estrogen.value,
                    "recent_trauma_4w": chk_trauma.value,
                    "unilateral_leg_swelling": chk_unilat.value,
                    # Wells-DVT details
                    "paresis": dvt_paresis.value,
                    "plaster_cast": dvt_paresis.value,
                    "bedridden_3d": dvt_bed_surg.value,
                    "surgery_12w": dvt_bed_surg.value,
                    "deep_vein_tenderness": dvt_tender.value,
                    "entire_leg_swollen": dvt_entire.value,
                    "calf_swelling_gt3cm": dvt_calf3.value,
                    "pitting_edema": dvt_edema.value,
                    "collateral_nonvaricose": dvt_collat.value,
                    "prev_dvt": dvt_prev.value,
                    "alt_dx_as_likely": dvt_alt_dx.value,
                })

                bundle = phase2_bundle(vitals, labs, ctx)

                # ML risk (only if bridge loaded)
                if "predict_one" in globals() and "FEAT" in globals():
                    feats = FEAT
                    row = {f: labs.get(f, vitals.get(f, ctx.get(f, 0.0))) for f in feats}
                    res = predict_one(row)
                    bundle["one_liners"].append(
                        f"ML risk p={res['p']:.3f} → {'ALERT' if res['y'] else 'ok'} (thr={res['thr']:.3f})"
                    )
                    try:
                        _append_event({"type": "ml_score", "keys": list(row.keys()), "res": res})
                    except Exception:
                        pass
                else:
                    bundle["one_liners"].append("ML risk: (bridge not loaded)")

                _append_event({"type":"phase2_bundle","bundle": bundle})
                print("\n".join(bundle["one_liners"]))
                print("\nLogged to:", EVENT_LOG_PATH)

        btn_apply_hl7.on_click(on_apply)
        btn_compute.on_click(on_compute)

        display(W.VBox([
            W.HTML("<b>Phase-2 calculators — optimized UI (PE/DVT/PERC, PESI/sPESI, SIRS/Sepsis3, HEART, GBS, Child-Pugh)</b>"),
            W.HBox([vitals_in, labs_in]),
            W.HBox([heart_hist, heart_ecg, heart_risk]),
            W.HTML("<b>PE / PERC flags</b>"),
            W.HBox([chk_pe_most, chk_dvt_signs, chk_immob, chk_prev_vte, chk_hemo, chk_cancer]),
            W.HBox([chk_estrogen, chk_trauma, chk_unilat]),
            W.HTML("<b>Wells-DVT flags</b>"),
            W.HBox([dvt_cancer, dvt_paresis, dvt_bed_surg, dvt_tender, dvt_entire]),
            W.HBox([dvt_calf3, dvt_edema, dvt_collat, dvt_prev, dvt_alt_dx]),
            ctx_in,
            W.HBox([btn_compute, btn_apply_hl7]),
            W.HTML("<b>HL7 Labs (Trop / D-Dimer / INR / Bili / Albumin)</b>"),
            hl7_in,
            out
        ]))
    except Exception as e:
        print("Phase-2 UI unavailable:", e)
else:
    # Text-mode smoke test (runs if RUN_UI=False)
    vit = {"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97,
           "heart_history":0,"heart_ecg":0,"heart_risk":0}
    lab = {"d_dimer":780,"d_dimer_unit":"μg/L FEU",
           "troponin_series":[{"time":"2025-08-20T05:00:00Z","value":18,"unit":"ng/L"},{"time":"2025-08-20T09:00:00Z","value":78,"unit":"ng/L"}],
           "creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20,"bilirubin":2.1,"inr":1.9}
    ctx = {"pregnant": True, "suspected_infection": True, "vasopressors": False, "pe_most_likely": True, "dvt_signs": False,
           "ascites":"mild","encephalopathy":"1-2"}
    b = phase2_bundle(vit, lab, ctx)
    _append_event({"type":"phase2_bundle_smoke","bundle": b})
    print("\n".join(b["one_liners"]))
    print("Logged to:", CONFIG.get("EVENT_LOG_PATH", "/mnt/data/event_log.jsonl"))


In [25]:
# Force-sync all "Context JSON" textareas and the reference used by YEARS
import json, ipywidgets as W, gc

def _update_all_context_widgets(flag=True):
    new_val = None
    # If a ctx_in exists in this scope, start from it
    try:
        d = json.loads(ctx_in.value or "{}")
    except Exception:
        d = {}
    d["pregnant"] = bool(flag)
    new_val = json.dumps(d, ensure_ascii=False)

    # Update any Textarea whose description starts with "Context JSON"
    n = 0
    for obj in gc.get_objects():
        try:
            if isinstance(obj, W.Textarea) and (obj.description or "").startswith("Context JSON"):
                obj.value = new_val
                n += 1
        except Exception:
            pass
    print(f"Updated {n} Context JSON widget(s) →", new_val)

    # Make sure the YEARS cell reads the same widget reference
    global ctx_src  # used inside the YEARS cell handler
    try:
        ctx_src = ctx_in
        print("ctx_src → ctx_in (bound)")
    except NameError:
        print("ctx_in not in scope; YEARS will still read its own ctx_src if present.")

_update_all_context_widgets(flag=True)


Updated 1 Context JSON widget(s) → {"pregnant": true, "suspected_infection": true, "vasopressors": false, "pe_most_likely": true, "dvt_signs": false, "ascites": "mild", "encephalopathy": "1-2"}
ctx_src → ctx_in (bound)


In [26]:

# === Pregnancy-adapted YEARS pathway (drop-in) ===
# Uses the Phase-2 panel's Context/Labs if available; otherwise exposes its own minimal text areas.
# Logs to CONFIG['EVENT_LOG_PATH'] as {"type": "pregnancy_years", ...}

from datetime import datetime, timezone
import json

def assess_pregnancy_years(ctx, labs):
    # Preg-adapted YEARS: three items — DVT signs, hemoptysis, 'PE most likely'.
    # If none present → D-dimer threshold 1000 μg/L FEU; else threshold 500 μg/L FEU.
    preg = bool(ctx.get("pregnant", False))
    items = {
        "dvt_signs": bool(ctx.get("dvt_signs", False)),
        "hemoptysis": bool(ctx.get("hemoptysis", False)),
        "pe_most_likely": bool(ctx.get("pe_most_likely", False)),
    }
    count = sum(int(v) for v in items.values())
    d_val = None
    for k in ("d_dimer","ddimer","d-dimer","d_dimer_feu_ug_l"):
        if k in labs and labs[k] not in (None, ""):
            try:
                d_val = float(str(labs[k]).replace(",", "."))
                break
            except Exception:
                pass
    d_unit = (labs.get("d_dimer_unit") or "μg/L FEU").lower()
    if d_val is None:
        return {"applies": preg, "needs_imaging": True, "reason": "missing D-dimer", "items": items, "items_count": count}
    d_ug = float(d_val)*1000.0 if "mg/l" in d_unit else float(d_val)
    thr = 1000.0 if count==0 else 500.0
    ruleout = bool(d_ug < thr)
    return {
        "applies": preg, "items": items, "items_count": count,
        "d_dimer_ug_l": d_ug, "threshold_ug_l": thr, "rule_out": ruleout,
        "note": "If DVT signs present, compression ultrasound is recommended before applying YEARS."
    }

# UI that reuses Phase-2 widget inputs if present (vitals_in / labs_in / ctx_in).
try:
    _probe = vitals_in  # noqa: F401
    _has_phase2_ui = True
except NameError:
    _has_phase2_ui = False

if CONFIG.get("RUN_UI", False):
    try:
        import ipywidgets as W
        if _has_phase2_ui:
            labs_src = labs_in
            ctx_src  = ctx_in
            src_note_ctx = W.HTML('<i>Using Context JSON from Phase-2 panel</i>')
            src_note_labs = W.HTML('<i>Using Labs JSON from Phase-2 panel</i>')
        else:
            labs_src = W.Textarea(value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU"}', description="Labs JSON", layout=W.Layout(width="100%", height="80px"))
            ctx_src  = W.Textarea(value='{"pregnant": true, "pe_most_likely": true, "dvt_signs": false, "hemoptysis": false}', description="Context JSON", layout=W.Layout(width="100%", height="80px"))
            src_note_ctx = src_note_labs = W.HTML('')
        out = W.Output()
        btn = W.Button(description="Compute Pregnancy YEARS + Log")
        warn = W.HTML("<small><b>Note:</b> PERC is not validated in pregnancy; use the YEARS pathway below.</small>")

        def _on_click(_):
            with out:
                out.clear_output()
                try:
                    labs = json.loads(labs_src.value if hasattr(labs_src, 'value') else labs_src)
                    ctx  = json.loads(ctx_src.value if hasattr(ctx_src, 'value') else ctx_src)
                except Exception as e:
                    print("[parse error]", e); return
                res = assess_pregnancy_years(ctx, labs)
                if not res.get("applies", False):
                    print("Pregnancy YEARS: not applicable (pregnant flag is false).")
                else:
                    msg = f"Pregnancy YEARS: items={res['items_count']} → D-dimer thr {int(res['threshold_ug_l'])} μg/L; value {int(res['d_dimer_ug_l'])} → "
                    msg += ("RULE-OUT" if res["rule_out"] else "IMAGING")
                    print(msg)
                try:
                    _append_event({"type":"pregnancy_years", "result": res})
                    print("Logged to:", CONFIG.get("EVENT_LOG_PATH"))
                except Exception as e:
                    print("[log error]", e)

        btn.on_click(_on_click)

        display(W.VBox([
            W.HTML("<b>Pregnancy-adapted YEARS pathway</b>"),
            warn,
            src_note_ctx, src_note_labs,
            btn, out
        ]))
    except Exception as e:
        print("YEARS UI unavailable:", e)

# Text-mode smoke if RUN_UI is off
if not CONFIG.get("RUN_UI", False):
    labs = {"d_dimer": 780, "d_dimer_unit": "μg/L FEU"}
    ctx = {"pregnant": True, "pe_most_likely": True, "dvt_signs": False, "hemoptysis": False}
    res = assess_pregnancy_years(ctx, labs)
    msg = f"Pregnancy YEARS (text-mode): items={res.get('items_count',0)}; thr={int(res.get('threshold_ug_l',0))} μg/L; value={int(res.get('d_dimer_ug_l',0))} → "
    msg += ("RULE-OUT" if res.get("rule_out") else "IMAGING")
    print(msg)
    try:
        _append_event({"type":"pregnancy_years_smoke", "result": res})
        print("Logged to:", CONFIG.get("EVENT_LOG_PATH"))
    except Exception as e:
        print("[log error]", e)


In [27]:
!tail -n 5 {CONFIG['EVENT_LOG_PATH']}


{"ts": "2025-08-31T18:49:04.127109Z", "type": "model_loaded", "bundle": "/kaggle/working/ed_phase2_model_thr_patched(1).joblib", "thr": 0.9988444286248084, "n_features": 16}


In [28]:

# Conference Demo: one-click run (synthetic ML + meds import + ICU ETA)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, json, pandas as pd, importlib.util
        run_btn = W.Button(description="Run Full Demo", button_style="success")
        out = W.Output()

        def _run(_):
            with out:
                out.clear_output()
                # 1) Synthetic ML pipeline (if configured)
                try:
                    CONFIG["RUN_SYNTH"] = True
                    print("[1/3] Running synthetic ML pipeline…")
                    # reuse the run_synth_pipeline if defined
                    try:
                        res = run_synth_pipeline(save_csv=False)
                    except NameError:
                        print("run_synth_pipeline not loaded in this kernel; skip")
                    else:
                        print("  →", res)
                except Exception as e:
                    print("Synthetic pipeline error:", e)

                # 2) ICU ETAs (if ICU panel code is present)
                try:
                    from datetime import datetime, timezone
                    # create a tiny scenario: set one unit to open in 30 min
                    state = {"timestamp": datetime.now(timezone.utc).isoformat(),
                             "units": [{"name":"1A Neurochirurgische Intensivstation","capacity":12,"occupied":12,"discharge_eta_minutes":[30,120]},
                                       {"name":"H2b Intensivstation Gefäß- und Herzmedizin","capacity":8,"occupied":7,"discharge_eta_minutes":[]} ]}
                    Path("/mnt/data/icu_status.json").write_text(json.dumps(state, ensure_ascii=False, indent=2))
                    print("[2/3] ICU sample state saved → /mnt/data/icu_status.json")
                except Exception as e:
                    print("ICU demo setup error:", e)

                # 3) Meds import demo
                try:
                    print("[3/3] Parsing demo BMP for patient SYN-DEMO-1…")
                    meds, warns = process_bmp_text("SYN-DEMO-1", DEMO_BMP)
                    print(f"  → {len(meds)} meds, {len(warns)} warnings; events appended.")
                except Exception as e:
                    print("Meds demo error:", e)

                # Tail event log
                try:
                    p = Path(CONFIG["EVENT_LOG_PATH"])
                    if p.exists():
                        lines = p.read_text().strip().splitlines()[-10:]
                        print("\nEvent log tail:")
                        for ln in lines:
                            print(" ", ln[:200])
                except Exception as e:
                    print("Log tail error:", e)

        run_btn.on_click(_run)
        display(W.VBox([run_btn, out]))
    except Exception as e:
        print("Demo UI unavailable:", e)
else:
    print("Demo UI deferred… set CONFIG['RUN_UI']=True.")


In [29]:

# === Medication Plan Import + Checks (inline, no sidecars) ===
from __future__ import annotations
from typing import Dict, Any, List, Optional, Tuple
from pathlib import Path
from datetime import datetime, timezone
import json, re

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# Minimal ATC/name map for demo (extendable)
_ATC_MAP = {
    "amoxicillin": {"atc": "J01CA04", "class": "penicillin"},
    "ibuprofen": {"atc": "M01AE01", "class": "nsaid"},
    "warfarin": {"atc": "B01AA03", "class": "coumarin"},
    "ramipril": {"atc": "C09AA05", "class": "ace"},
    "spironolacton": {"atc": "C03DA01", "class": "aldosterone_antagonist"},
    "metoprolol": {"atc": "C07AB02", "class": "beta_blocker"},
    "simvastatin": {"atc": "C10AA01", "class": "statin"},
    "clarithromycin": {"atc": "J01FA09", "class": "macrolide"},
    "azithromycin": {"atc": "J01FA10", "class": "macrolide"},
}

# Demo interaction rules (replace via MED_RULES_PATH for your own rules)
_DEMO_RULES = [
    {"a": "ibuprofen", "b": "warfarin", "severity": "major", "message": "Bleeding risk (NSAID + warfarin)"},
    {"a": "ramipril", "b": "spironolacton", "severity": "moderate", "message": "Hyperkalemia risk (ACE + aldosterone antagonist)"},
    {"a": "amoxicillin", "b": "warfarin", "severity": "moderate", "message": "Antibiotic may potentiate warfarin effect"},
    {"a": "clarithromycin", "b": "simvastatin", "severity": "major", "message": "Rhabdomyolysis risk (CYP3A4 inhibition)"},
]

_ROUTES = ["p.o.", "po", "i.v.", "iv", "i.m.", "im", "s.c.", "sc", "inhalativ", "topisch", "nasal", "otic", "ophthalmic"]
_FREQ_WORDS = ["morgens", "mittags", "abends", "nachts"]
_FREQ_PAT = re.compile(r"\b(\d+)[-/.](\d+)[-/.](\d+)\b")
_DOSE_PAT = re.compile(r"(\d+(?:[.,]\d+)?)\s*(mg|g|mcg|µg|ml|IE|Einheiten)\b", flags=re.I)

def _normalize_name(s: str) -> str:
    s = s.strip().lower()
    s = s.replace("ä","ae").replace("ö","oe").replace("ü","ue").replace("ß","ss")
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

def _map_to_atc(name_norm: str) -> Dict[str, Any]:
    for key, meta in _ATC_MAP.items():
        if key in name_norm:
            return {"name_norm": key, **meta}
    return {"name_norm": name_norm, "atc": None, "class": None}

def parse_med_line(line: str) -> Optional[Dict[str, Any]]:
    raw = line.strip()
    if not raw or raw.startswith("#"):
        return None
    name = raw.split(",")[0].split("  ")[0]  # up to comma or double spaces
    name_norm = _normalize_name(name)
    dose_val, dose_unit = None, None
    m = _DOSE_PAT.search(raw)
    if m:
        dose_val = float(m.group(1).replace(",", "."))
        dose_unit = m.group(2).lower()
    freq = None
    m = _FREQ_PAT.search(raw)
    if m:
        freq = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    else:
        words = [w for w in _FREQ_WORDS if w in raw.lower()]
        if words:
            freq = ",".join(words)
    route = None
    for r in _ROUTES:
        if r in raw.lower():
            route = r
            break
    prn = bool(re.search(r"\b(prn|bei bedarf)\b", raw, flags=re.I))
    atc_meta = _map_to_atc(name_norm)
    return {
        "raw": raw,
        "name": name.strip(),
        "name_norm": atc_meta["name_norm"],
        "atc": atc_meta["atc"],
        "class": atc_meta["class"],
        "dose_value": dose_val,
        "dose_unit": dose_unit,
        "frequency": freq,
        "route": route,
        "prn": prn,
    }

def parse_med_text(text: str) -> List[Dict[str, Any]]:
    meds = []
    for line in text.splitlines():
        rec = parse_med_line(line)
        if rec:
            meds.append(rec)
    return meds

def load_rules(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, list):
                return js
        except Exception:
            pass
    return list(_DEMO_RULES)

def load_allergies(path: str) -> List[str]:
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and "allergies" in js and isinstance(js["allergies"], list):
                return [str(x) for x in js["allergies"]]
        except Exception:
            pass
    return []

def check_interactions(meds: List[Dict[str, Any]], rules: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    names = [m["name_norm"] for m in meds]
    for r in rules:
        a, b = r["a"], r["b"]
        if (a in names and b in names) or (b in names and a in names):
            out.append({"type": "med_interaction_warning", **r})
    return out

def check_allergies(meds: List[Dict[str, Any]], allergy_terms: List[str]) -> List[Dict[str, Any]]:
    out = []
    terms = [t.lower() for t in allergy_terms]
    for m in meds:
        # direct name match
        if any(t in m["name"].lower() for t in terms):
            out.append({"type": "med_allergy_warning", "med": m["name"], "match": "name"})
            continue
        # conservative penicillin class heuristic
        if any("penicillin" in t for t in terms):
            if m["class"] == "penicillin" or m["name"].lower().endswith("cillin"):
                out.append({"type": "med_allergy_warning", "med": m["name"], "match": "class_penicillin"})
    return out

def ocr_file(path: str) -> Optional[str]:
    p = Path(path)
    if not p.exists():
        return None
    try:
        from PIL import Image
        import pytesseract
        if p.suffix.lower() in [".png",".jpg",".jpeg",".tif",".tiff"]:
            return pytesseract.image_to_string(Image.open(p))
        if p.suffix.lower() == ".pdf":
            try:
                from pdf2image import convert_from_path
                pages = convert_from_path(str(p))
                text = ""
                for img in pages:
                    text += pytesseract.image_to_string(img) + "\n"
                return text
            except Exception:
                return None
    except Exception:
        return None
    return None

def emit_meds_events(patient_id: str, meds: List[Dict[str, Any]], warnings: List[Dict[str, Any]]):
    _append_event({"type": "med_plan_import", "patient_id": patient_id, "n_meds": len(meds)})
    for m in meds:
        _append_event({"type": "med_entry", "patient_id": patient_id, **m})
    for w in warnings:
        _append_event({**w, "patient_id": patient_id})

if CONFIG.get("RUN_MEDS"):
    print("Medication import/checks enabled. Use the UI panel (if RUN_UI=True) or call the functions above.")
else:
    print("Deferred… set CONFIG['RUN_MEDS']=True to enable medication import & checks.")

# Optional UI
if CONFIG.get("RUN_UI") and CONFIG.get("RUN_MEDS"):
    try:
        import ipywidgets as W
        import pandas as pd
        pid = W.Text(description="Patient ID", placeholder="e.g., UKE-12345")
        path = W.Text(description="Scan path", placeholder="/mnt/data/scan.pdf (optional)")
        ocr_btn = W.Button(description="Run OCR")
        ta = W.Textarea(description="Plan text", layout=W.Layout(width="100%", height="180px"))
        parse_btn = W.Button(description="Parse & Check", button_style="primary")
        out = W.Output()

        def on_ocr(_):
            with out:
                out.clear_output()
                txt = ocr_file(path.value.strip())
                if txt:
                    ta.value = txt
                    print("OCR complete.")
                else:
                    print("OCR unavailable or failed. Paste the text instead.")

        def on_parse(_):
            with out:
                out.clear_output()
                text = ta.value.strip()
                if not text:
                    print("No text provided."); return
                meds = parse_med_text(text)
                rules = load_rules(CONFIG["MED_RULES_PATH"])
                allergies = load_allergies(CONFIG["ALLERGIES_PATH"])
                warnings = check_interactions(meds, rules) + check_allergies(meds, allergies)
                emit_meds_events(pid.value or "DEMO", meds, warnings)
                if meds:
                    display(pd.DataFrame(meds))
                else:
                    print("No medications parsed.")
                if warnings:
                    print("\nWarnings:")
                    display(pd.DataFrame(warnings))
                else:
                    print("\nNo interaction/allergy warnings.")

        ocr_btn.on_click(on_ocr)
        parse_btn.on_click(on_parse)
        display(W.VBox([pid, path, W.HBox([ocr_btn, parse_btn]), ta, out]))
    except Exception as e:
        print("Meds UI unavailable:", e)


Medication import/checks enabled. Use the UI panel (if RUN_UI=True) or call the functions above.


In [30]:

# === Conference Demo: one-click walkthrough ===
from pathlib import Path
from datetime import datetime, timezone
import json

def _tail_events(n=12):
    p = Path(CONFIG["EVENT_LOG_PATH"])
    if not p.exists():
        return []
    lines = p.read_text().splitlines()
    return [json.loads(x) for x in lines[-n:]] if lines else []

if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd

        btn = W.Button(description="Conference Demo → Run Full Demo", button_style="success")
        out = W.Output()

        def run_demo(_):
            with out:
                out.clear_output()
                print("Running conference demo…")

                # Step 1: Synthetic ML pipeline (if enabled)
                if CONFIG.get("RUN_SYNTH"):
                    try:
                        res = run_synth_pipeline(save_csv=False)
                        print("[SYNTH]", res)
                    except Exception as e:
                        print("[SYNTH ERROR]", e)
                else:
                    print("[SYNTH] Skipped (set RUN_SYNTH=True to include)")

                # Step 2: ICU snapshot → next bed in 30 min on one unit
                icu = {
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "units": [
                        {"name":"1C Interdisziplinäre Intensivstation","capacity":12,"occupied":12,"discharge_eta_minutes":[30,120,240]},
                        {"name":"1G Internistische Intensivstation","capacity":12,"occupied":11,"discharge_eta_minutes":[60]}
                    ]
                }
                Path("/mnt/data/icu_status.json").write_text(json.dumps(icu, ensure_ascii=False, indent=2))
                print("[ICU] Snapshot saved → next bed on 1C in 30 min")

                # Step 3: Medication plan demo (paste-mode)
                demo_text = """\
Amoxicillin 500 mg 1-0-1 p.o.
Ibuprofen 400 mg 1-1-1 p.o. bei Bedarf
Warfarin 5 mg 1-0-0 p.o.
Ramipril 5 mg 1-0-0 p.o.
Spironolacton 25 mg 0-0-1 p.o.
"""
                meds = parse_med_text(demo_text)
                rules = load_rules(CONFIG["MED_RULES_PATH"])
                allergies = ["Penicillin"]  # demo allergy to trigger warning with Amoxicillin
                warnings = check_interactions(meds, rules) + check_allergies(meds, allergies)
                emit_meds_events("DEMO-123", meds, warnings)
                print("[MEDS] Parsed", len(meds), "entries; warnings:", len(warnings))

                # Show event log tail
                tail = _tail_events(12)
                if tail:
                    display(pd.DataFrame(tail))
                else:
                    print("No events yet.")

        btn.on_click(run_demo)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("Demo UI unavailable:", e)
else:
    print("Demo UI deferred… set CONFIG['RUN_UI']=True.")


In [31]:

# --- UI helpers (streamlit-optional, auto-appended) ---
try:
    import streamlit as st
except ModuleNotFoundError:
    st = None

def run_ui_header_controls():
    if st is None:
        return 120, 120
    c0, c1, c2, c3 = st.columns([2,2,2,2])
    with c0:
        thresh = st.number_input("Overdue threshold (min)", min_value=5, max_value=720, value=120, step=5)
    with c1:
        vitals_thresh = st.number_input("Vitals overdue (min)", min_value=5, max_value=720, value=120, step=5)
    with c2:
        if st.button("Refresh"):
            st.experimental_rerun()
    return thresh, vitals_thresh

def run_ui_equipment_panel(tracker, overdue_thresh_min: float):
    if st is None:
        print("[UI disabled] Equipment panel skipped."); return
    import pandas as pd
    st.header("Equipment")
    eq_df = tracker.equipment_status()
    s1, s2 = st.columns([2,1])
    with s1:
        q = st.text_input("Find equipment (ID / name / location / status)", "")
        filt = tracker.find_equipment(q) if q else eq_df
        st.dataframe(filt, use_container_width=True, height=260)
    with s2:
        overdue = tracker.overdue_equipment(int(overdue_thresh_min))
        st.subheader("Overdue")
        if overdue.empty:
            st.write("None")
        else:
            st.dataframe(overdue[["equip_id","name","location","last_seen","age_min"]],
                         use_container_width=True, height=200)
    st.markdown("**Update location / log move**")
    mc1, mc2, mc3, mc4 = st.columns([2,2,2,1])
    with mc1:
        sel_id = st.selectbox("Equipment ID", [""] + sorted(list(getattr(eq_df, "equip_id", []))))
    with mc2:
        loc_from = st.text_input("From", "")
    with mc3:
        loc_to = st.text_input("To", "")
    with mc4:
        if st.button("Log move") and sel_id and loc_to:
            tracker.log_move(sel_id, loc_from, loc_to)
            st.success(f"Move logged: {sel_id} → {loc_to}")

def run_ui_qr_panel(tracker):
    if st is None:
        print("[UI disabled] QR panel skipped."); return
    st.header("QR")
    qr_col1, qr_col2 = st.columns([2,2])
    with qr_col1:
        qr_txt = st.text_input("QR payload to generate", "")
        if st.button("Generate QR") and qr_txt:
            path = tracker.make_qr(qr_txt)
            st.write("QR saved to:", path)
    with qr_col2:
        st.write("Scan and update location")
        f = st.file_uploader("Upload QR image", type=["png","jpg","jpeg","webp"])
        manual_payload = st.text_input("Manual payload (fallback if decoding fails)", "")
        new_loc = st.text_input("New location (after scan)", "")
        if st.button("Scan & Update"):
            equip_payload = None
            if f is not None:
                equip_payload = tracker.decode_qr_bytes(f.read())
            if not equip_payload and manual_payload:
                equip_payload = manual_payload
            if equip_payload and new_loc:
                equip_id = equip_payload
                if "id=" in equip_payload:
                    try:
                        equip_id = equip_payload.split("id=",1)[1].split("&",1)[0]
                    except Exception:
                        equip_id = equip_payload
                tracker.log_move(str(equip_id), "", new_loc)
                st.success(f"Updated via payload. {equip_id} → {new_loc}")
            elif not new_loc:
                st.error("Provide a new location.")
            else:
                st.error("No QR payload detected (image or manual).")

def run_ui_sop_panel():
    if st is None:
        print("[UI disabled] SOP panel skipped."); return
    with st.expander("SOP auto-pull and flows", expanded=False):
        if st.button("Refresh SOPs from sop-notaufnahme.de"):
            res = refresh_sop_registry(CONFIG, base_url="https://sop-notaufnahme.de/sop/")
            st.write(res)
        flows = load_priority_flows("/mnt/data/priority_flows.json")
        if flows:
            keys = sorted(list(flows.keys()))
            pickf = st.selectbox("Show flow", [""] + keys)
            if pickf:
                flow = flows[pickf]
                st.subheader(flow.get("title", pickf))
                nodes = flow.get("nodes", []); edges = flow.get("edges", [])
                try:
                    import matplotlib.pyplot as plt
                    fig = plt.figure()
                    pos = {n["id"]:(i, 0) for i,n in enumerate(nodes)}
                    for n in nodes:
                        x,y = pos[n["id"]]; plt.scatter([x],[y])
                        plt.text(x,y+0.05,n.get("label", n["id"]), ha="center", rotation=45)
                    for a,b in edges:
                        xa,ya = pos.get(a,(0,0)); xb,yb = pos.get(b,(0,0))
                        plt.plot([xa,xb],[ya,yb])
                    plt.axis("off"); plt.title(flow.get("title", pickf))
                    st.pyplot(fig)
                except Exception:
                    st.info("Graph display unavailable; showing list instead.")
                    st.write(edges)

def run_ui_actions_and_critic(get_state, get_actions, critic, vitals_thresh_min: float):
    if st is None:
        print("[UI disabled] Actions & Critic skipped."); return
    import pandas as pd, numpy as np
    st.header("SOPs")
    st.header("Actions & Critic")
    state = get_state()
    if hasattr(state, "feature_dict"):
        feats = state.feature_dict()
        since_v = feats.get("since_vitals_min", None)
        if since_v is not None:
            if since_v > float(vitals_thresh_min):
                st.error(f"Lingering patient: since_vitals_min={since_v:.0f} > {int(vitals_thresh_min)}")
            else:
                st.success(f"Vitals recently checked: {since_v:.0f} min (≤ {int(vitals_thresh_min)})")
    if st.button("Mark vitals now") and hasattr(state, "touch_now"):
        import pandas as pd
        state.touch_now(pd.Timestamp.utcnow())
        st.success("Vitals timestamp updated.")
    actions = get_actions(state) or []
    if not actions:
        st.info("No actions available."); return
    p, benefit, burden = critic.score(state, actions)
    view = pd.DataFrame({
        "id":[a.get("id") for a in actions],
        "label":[a.get("label") for a in actions],
        "p_accept":np.round(p,3),
        "benefit":np.round(benefit,3),
        "burden":np.round(burden,3)
    }).sort_values(["p_accept","benefit"], ascending=[False, False])
    st.dataframe(view, use_container_width=True, height=240)

def run_ui_movement_analytics(tracker):
    if st is None:
        print("[UI disabled] Movement analytics skipped."); return
    st.header("Equipment Movement Analytics")
    stats = tracker.movement_stats()
    per_eq = stats.get("moves_per_equipment"); routes = stats.get("routes")
    if per_eq is None or getattr(per_eq, "empty", True):
        st.info("No movement data yet."); return
    st.subheader("Moves per equipment")
    st.dataframe(per_eq, use_container_width=True, height=240)
    try:
        import matplotlib.pyplot as plt
        fig = plt.figure()
        x = per_eq["equip_id"].astype(str).tolist(); y = per_eq["moves"].tolist()
        plt.bar(x,y); plt.xticks(rotation=45, ha="right"); plt.title("Moves per Equipment")
        st.pyplot(fig)
    except Exception:
        pass
    st.subheader("Top routes")
    if routes is not None:
        st.dataframe(routes, use_container_width=True, height=200)


In [32]:

# --- Wrapper (streamlit-optional) ---
def run_ui(tracker, get_state, get_actions, critic):
    try:
        st  # from helpers cell
    except NameError:
        st = None  # noqa: F841

    if st is None:
        # notebook fallback
        import pandas as pd, numpy as np
        from IPython.display import display
        print("UI — notebook fallback (no Streamlit)")
        overdue_thresh, vitals_thresh = run_ui_header_controls()  # defaults
        try:
            eq_df = tracker.equipment_status()
            print("\nEquipment (sample):"); display(eq_df.head(20))
        except Exception:
            pass
        state = get_state()
        actions = get_actions(state) or []
        if actions:
            p, benefit, burden = critic.score(state, actions)
            view = pd.DataFrame({
                "id":[a.get("id") for a in actions],
                "label":[a.get("label") for a in actions],
                "p_accept":np.round(p,3),
                "benefit":np.round(benefit,3),
                "burden":np.round(burden,3),
            }).sort_values(["p_accept","benefit"], ascending=[False, False])
            print("\nActions & Critic:"); display(view)
        else:
            print("\nNo actions available.")
        if hasattr(tracker, "movement_stats"):
            stats = tracker.movement_stats()
            per_eq = stats.get("moves_per_equipment"); routes = stats.get("routes")
            if per_eq is not None:
                print("\nMoves per Equipment:"); display(per_eq)
            if routes is not None:
                print("\nTop Routes:"); display(routes)
        print("ui ready")
        return

    # streamlit path
    import pandas as pd, numpy as np
    st.set_page_config(page_title="ED Tracker — Full", layout="wide")
    st.title("ED Tracker — Core Ops (Full)")

    overdue_thresh, vitals_thresh = run_ui_header_controls()
    run_ui_equipment_panel(tracker, overdue_thresh)
    run_ui_qr_panel(tracker)
    run_ui_sop_panel()

    sop_q = st.text_input("Search SOPs (id/title/keywords)", "")
    sop_hits = tracker.search_sop(sop_q)
    if getattr(sop_hits, "empty", True):
        st.info("No SOPs found.")
    else:
        st.dataframe(sop_hits[["sop_id","title","version","status"]], use_container_width=True, height=220)
        pick_opts = [""] + sop_hits["sop_id"].astype(str).tolist()
        pick = st.selectbox("Open SOP", pick_opts)
        if pick:
            row = sop_hits[sop_hits["sop_id"].astype(str)==pick].iloc[0]
            pdf = row.get("pdf_path","")
            if pdf: st.write("PDF path:", pdf)
            if "checklist" in sop_hits.columns and isinstance(row.get("checklist", None), str) and row["checklist"].strip():
                st.subheader("Checklist")
                steps = [s.strip() for s in row["checklist"].split("|") if s.strip()]
                completed = []
                for i, step in enumerate(steps, 1):
                    if st.checkbox(f"{i}. {step}", key=f"sop_{pick}_{i}"):
                        completed.append(i)
                st.caption(f"Completed {len(completed)}/{len(steps)} steps")

    run_ui_actions_and_critic(get_state, get_actions, critic, vitals_thresh)
    run_ui_movement_analytics(tracker)
    st.caption("ui ready")


In [33]:

# --- Streamlit-aware SPC/UI launcher (notebook-safe) ---
import importlib.util as _ilu
_HAS_STREAMLIT = _ilu.find_spec("streamlit") is not None

def _run_spc_notebook():
    import pandas as pd
    from IPython.display import display
    print("SPC — notebook fallback (no Streamlit)")
    for name in ("ed","dx","meds"):
        df = globals().get(name)
        if isinstance(df, pd.DataFrame) and not df.empty:
            print(f"\n{name.upper()} sample:"); display(df.head(50))

# Resolve SPC entrypoint
if "run_spc" not in globals():
    if "spc_dashboard" in globals() and _HAS_STREAMLIT:
        run_spc = spc_dashboard
    else:
        run_spc = _run_spc_notebook

try:
    if CONFIG.get("RUN_SPC", True):   # demo: default ON
        run_spc()
        print("SPC started ✓")

    if CONFIG.get("RUN_UI", True):    # demo: default ON
        if _HAS_STREAMLIT and "run_ui" in globals():
            _append_event({"type": "ui_start"}) if "_append_event" in globals() else None
            _ = run_ui(tracker=tracker, get_state=get_state, get_actions=get_actions, critic=critic)
            print("UI started (Streamlit) ✓")
        else:
            print("UI skipped (Streamlit not available in this job).")
except Exception as e:
    print("UI/SPC start failed:", repr(e))
    raise


SPC — notebook fallback (no Streamlit)
SPC started ✓
UI skipped (Streamlit not available in this job).


In [34]:
# === quick bundle smoke (demo-safe) ===
from pathlib import Path
import pickle

bp = CONFIG.get("MODEL_BUNDLE_PATH")
assert bp and Path(bp).exists(), f"Bundle path missing: {bp}"

# Load bundle (prefer joblib, fall back to pickle)
try:
    import joblib  # usually available in Kaggle image
    bundle = joblib.load(bp)
except Exception:
    with open(bp, "rb") as f:
        bundle = pickle.load(f)

thr = bundle.get("threshold", None)
n_features = bundle.get("features", None)
feature_names = bundle.get("feature_names_", None) or bundle.get("features_names", None)

print(f"bundle ok → thr={thr} | features={n_features} | sample keys={list(bundle.keys())[:6]}")

# Optional: probe predict_one if present in THIS notebook
if "predict_one" in globals() and callable(predict_one):
    feats = {}
    if isinstance(feature_names, (list, tuple)):
        feats = {name: 0 for name in feature_names}
    try:
        proba, label = predict_one(feats)
        print("predict_one ok →", proba, label)
    except Exception as e:
        print("predict_one present but input mismatch:", repr(e))
else:
    print("predict_one not defined here (expected for demo notebook).")


bundle ok → thr=0.9988444286248084 | features=['Tag', 't_min', 'Triage', 'Leitsymptom', 'HF', 'MAP', 'ICU_Kap', 'Kap_veraltet', 't_norm', 'hat_Labor', 'Labor_ausstehend', 'hat_Roentgen', 'Roentgen_ausstehend', 'hat_CT', 'CT_ausstehend', 'naechste_Aktion'] | sample keys=['pipeline', 'calibrator', 'threshold', 'features', 'label', 'model_kind']
predict_one present but input mismatch: KeyError("None of [Index(['Tag', 't_min', 'Triage', 'Leitsymptom', 'HF', 'MAP', 'ICU_Kap',\n       'Kap_veraltet', 't_norm', 'hat_Labor', 'Labor_ausstehend',\n       'hat_Roentgen', 'Roentgen_ausstehend', 'hat_CT', 'CT_ausstehend',\n       'naechste_Aktion'],\n      dtype='object')] are in the [columns]")


In [35]:
# bundle loads? (quiet)
from pathlib import Path
import joblib, pickle
bp = CONFIG.get("MODEL_BUNDLE_PATH")
if bp and Path(bp).exists():
    try:
        _ = joblib.load(bp)
        print("bundle load: OK")
    except Exception:
        with open(bp, "rb") as f: _ = pickle.load(f)
        print("bundle load: OK (pickle)")
else:
    print("bundle load: skipped (no path)")


bundle load: OK
